In [ ]:
# INQUIRE_CSI – Contaminant Source Identification Pipeline
## Part 2 – INQUIRE_CSI_Pt2 (Contaminant Source Identification and Validation)
"""
Attributes the compounds of each sample to the most plausible sources (weighted set cover with priors) and validates the method.
Documentation: `docs/02_identification_validation.md`
"""


In [ ]:
# Reset environment to erase previous vars, dicts, DFs, etc.
%reset -f

from datetime import datetime

print(f"Environment reset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
# Imports & setup
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import logging
import hashlib
import random
import re
from tqdm.auto import tqdm
from tqdm.auto import trange
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import beta
import openpyxl
import time
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from collections import defaultdict
from functools import lru_cache

# Logging setup (equivalent to logger + appender_tee)
log_file = f"inquire_csi_pt2_log_{datetime.now().strftime('%y%m%d')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[
        logging.FileHandler(log_file, mode="w"),
        logging.StreamHandler()
    ]
)


# Helper functions
def majorsep():
    logging.info("=" * 100)

def minorsep():
    logging.info("-" * 100)

def percent(n, N):
    return f"{(100 * n / N):.1f}%" if N > 0 else "0%"

# Validation helpers
def check_file(path_obj, label):
    if not path_obj.exists():
        raise FileNotFoundError(f" - {label} missing: {path_obj}")
    logging.info(f" - Found file: {label}")

def check_columns(df, required_cols, df_name):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f" - {df_name} missing required columns: {missing}")
    logging.info(f" - {df_name}: required columns present")

# Preferred encodings (tried in order)
ENCODINGS = ["utf-8", "utf-8-sig", "cp1252", "cp1250", "latin1"]

# Read only the header row using multiple encodings. Supports CSV, TSV, XLSX and XLS.
def _read_header_only(path: Path):
    suffix = path.suffix.lower()
    if suffix in [".csv", ".tsv"]:
        sep = "\t" if suffix == ".tsv" else ","
        last_error = None
        for enc in ENCODINGS:
            try:
                df = pd.read_csv(path, nrows=0, sep=sep, encoding=enc)
                logging.info(f" - Header read using encoding '{enc}'")
                return list(df.columns)
            except UnicodeDecodeError as e:
                last_error = e
        raise RuntimeError(f"Unable to decode header using supported encodings.\nLast error:\n{last_error}")
    elif suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(path, nrows=0)
        return list(df.columns)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

# Check that all expected columns exist before loading the full dataset
def check_cols_before_load(path, expected_cols, dataset_name="dataset"):
    path = Path(path)
    try:
        cols = _read_header_only(path)
    except Exception as e:
        raise RuntimeError(f"[{dataset_name}] failed to read header:\n{e}")
    missing = [c for c in expected_cols if c not in cols]
    if missing:
        raise ValueError(
            f"[{dataset_name}] missing columns:\n"
            f"{missing}\n\n"
            f"Available columns:\n{cols}"
        )
    logging.info(f" - [{dataset_name}] column check passed")
    return True

# Read CSV safely using automatic encoding fallback
def safe_read_csv(path_obj, label, **kwargs):
    if "low_memory" not in kwargs:
        kwargs["low_memory"] = False
    check_file(path_obj, label)
    last_error = None
    for enc in ENCODINGS:
        try:
            df = pd.read_csv(path_obj, encoding=enc, **kwargs)
            logging.info(
                f" - Loaded {label} "
                f"(encoding={enc}) | "
                f"rows={len(df):,} cols={len(df.columns)}"
            )
            return df
        except UnicodeDecodeError as e:
            last_error = e
    raise RuntimeError(
        f" - Failed loading {label}. "
        f"None of the tested encodings worked.\n"
        f"Last error:\n{last_error}"
    )

# Read Excel safely      
def safe_read_excel(path_obj, label, **kwargs):
    check_file(path_obj, label)
    try:
        df = pd.read_excel(path_obj, **kwargs)
        logging.info(f" - Loaded {label} | rows={len(df)} cols={len(df.columns)}")
        return df
    except Exception as e:
        raise RuntimeError(f" - Failed loading {label}: {e}")

# Strip leading/trailing whitespace from selected string columns
def strip_whitespace_columns(df, columns, dataset_name="dataset"):
    existing_cols = [c for c in columns if c in df.columns]
    if not existing_cols:
        logging.info(f" - No whitespace-cleanup columns found in {dataset_name}")
        return df
    for col in existing_cols:
        # only process non-null values
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )
    logging.info(
        f" - Whitespace cleanup applied to {dataset_name} | "
        f"Columns: {', '.join(existing_cols)}"
    )
    return df


# Seed
GLOBAL_SEED = 1
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

majorsep()
logging.info(f"Imports & logging setup done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Check if necessary columns exist in input data files
majorsep()
logging.info(f"Checking if necessary columns exist in input data files | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Data folder & subfolders
DATA_DIR = Path("../data") # specify path to your working folder (default: 'data' folder of the repository)
if not DATA_DIR.exists(): raise FileNotFoundError(f"Working folder not found: {DATA_DIR.resolve()} → specify DATA_DIR")

# Switch for the household geo metadata priors
#  - True : household_metadata_priors.csv (in DATA_DIR/geo_metadata_outputs) is used to adjust the source priors per household
#  - False: no household metadata are used (geo multiplier = 1.0 for all sources and samples)
USE_GEO_PRIORS = True

STD_DIR = Path(DATA_DIR / "databases_standardized")
STD_DIR.mkdir(exist_ok=True)
MASK_DIR = Path(DATA_DIR / "masked_sources")
MASK_DIR.mkdir(exist_ok=True)
GEO_DIR = Path(DATA_DIR / "geo_metadata_outputs")
GEO_DIR.mkdir(exist_ok=True)
ALF_DIR = Path(DATA_DIR / "alpha_tuning")
ALF_DIR.mkdir(exist_ok=True)
VAL_DIR = Path(DATA_DIR / "validation_outputs")
VAL_DIR.mkdir(exist_ok=True)
OUT_DIR = Path(DATA_DIR / "set_cover_outputs")
OUT_DIR.mkdir(exist_ok=True)

# Check columns
check_cols_before_load(MASK_DIR / "sources_masked.csv", ["source_id", "source_name", "inchikey", "source_origin", "source_kind", "source_general_category", "source_harmonized", "source_class", "source_class_domain"], "sources_masked")
minorsep()
check_cols_before_load(MASK_DIR / "sample_universes.csv", ["inchikey"], "sample_universes")
minorsep()
check_cols_before_load(STD_DIR / "sources_meta_standardized.csv", ["source_id", "source_name", "source_origin", "source_kind", "source_general_category", "source_harmonized", "source_class", "source_class_domain"], "sources_meta")
minorsep()

# Use the name of the file from Pt1 (e.g., "dust_standardized.csv")
# and accounting for absence/presence of QC columns (e.g., "dust_standardized_qc_cols.csv")
check_cols_before_load(STD_DIR / "dust_standardized.csv", ["Feature Name", "inchikey"], "standardized sample dataset")
minorsep()

# "Map geo metadata modules to sources" cell checks that
# *_likelihood columns in the file match MODULE_WEIGHT and MODULE_SOURCE_MAP
if USE_GEO_PRIORS:
    check_cols_before_load(GEO_DIR / "household_metadata_priors.csv", ["household_id"], "geo_metadata_prior")
else:
    logging.info(f" - USE_GEO_PRIORS = False → household_metadata_priors.csv is not used")

minorsep()
logging.info(f"Done checking if necessary columns exist in input data files | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Load input data
majorsep()
logging.info(f"Loading input data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Read files
sources_masked_0 = safe_read_csv(MASK_DIR / "sources_masked.csv", "sources_masked", sep=",")
logging.info(f" - Loaded sources_masked | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

sample_universes_0 = safe_read_csv(MASK_DIR / "sample_universes.csv", "sample_universes", sep=",")
logging.info(f" - Loaded sample_universes | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

sources_meta_0 = safe_read_csv(STD_DIR / "sources_meta_standardized.csv", "sources_meta", sep=",")
logging.info(f" - Loaded sources_meta | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Use the name of the file from Pt1 (e.g., "dust_standardized.csv")
# and accounting for absence/presence of QC columns (e.g., "dust_standardized_qc_cols.csv")
dust_full_std_0 = safe_read_csv(STD_DIR / "dust_standardized.csv", "sample_data_standardized", sep=",")
logging.info(f" - Loaded standardized sample dataset, e.g., dust_standardized | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

logging.info(f"Done loading input data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Load household metadata priors
majorsep()
logging.info(f"Loading household geo metadata priors | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

if USE_GEO_PRIORS:
    metadata_prior_0 = safe_read_csv(GEO_DIR / "household_metadata_priors.csv", "geo_metadata_prior", sep=",")
    logging.info(f" - Loaded geo_metadata_prior | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
else: # empty table → no household has metadata → geo multiplier = 1.0 everywhere
    metadata_prior_0 = pd.DataFrame(columns=["household_id"])
    logging.info(f" - USE_GEO_PRIORS = False → no household geo metadata priors are used")
minorsep()

module_cols = [c for c in metadata_prior_0.columns if c.endswith("_likelihood")]

# Household key = 'LLNN' (LL = 2-letter sample code, NN = 2-digit sample number)
#  - Sample names can carry suffixes (e.g., 'FR01_LS2'); these are ignored when matching to the geo metadata
#  - household_key() is also used in the "Geo metadata Bayesian helper functions" and "Per-sample cover analysis" cells
#    (and for own household products), so it is defined even when USE_GEO_PRIORS = False

HOUSEHOLD_KEY_PATTERN = re.compile(r"^([A-Z]{2}\d{2})(?=_|$)")

def household_key(sample_or_household_id):
    if sample_or_household_id is None:
        return None
    s = str(sample_or_household_id).strip().upper()
    m = HOUSEHOLD_KEY_PATTERN.match(s)
    return m.group(1) if m else s # IDs not in LLNN format are returned unchanged (upper case, no spaces)

metadata_prior_0["household_key"] = metadata_prior_0["household_id"].map(household_key)

non_standard_ids = metadata_prior_0.loc[~metadata_prior_0["household_key"].astype(str).str.match(HOUSEHOLD_KEY_PATTERN), "household_id"].tolist() # astype(str) also works for an empty table
if non_standard_ids:
    logging.warning(f" - Geo metadata IDs not in LLNN format (kept as they are): {non_standard_ids[:20]}")

duplicated_keys = metadata_prior_0.loc[metadata_prior_0["household_key"].duplicated(keep=False), "household_id"].tolist()
if duplicated_keys:
    logging.warning(f" - Several geo metadata rows map to the same household (first row kept): {duplicated_keys[:20]}")

hh_metadata_lookup_0 = (
    metadata_prior_0
    .drop_duplicates(subset="household_key")
    .set_index("household_key")[module_cols]
    .to_dict("index")
)

logging.info(f" - Households: {len(hh_metadata_lookup_0):,}")
logging.info(f" - Modules   : {len(module_cols)}")

minorsep()
logging.info(f"Done loading household geo metadata priors | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Create copies of raw DFs
majorsep()
logging.info(f"Creating working copies of input DFs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

sources_meta = sources_meta_0.copy()
dust_full_std = dust_full_std_0.copy()
sources_masked = sources_masked_0.copy()
sample_universes_df = sample_universes_0.copy()
metadata_prior = metadata_prior_0.copy()
hh_metadata_lookup = hh_metadata_lookup_0.copy()

logging.info(f"Created working copies of input DFs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Precompute source metadata lookup
majorsep()
logging.info(f"Precomputing source metadata lookup | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

source_meta_unique = (sources_meta.sort_values("source_id").groupby("source_id", as_index=False).first())
source_meta_lookup = source_meta_unique.set_index("source_id")

logging.info(f" - source_meta_unique rows indexed: {len(source_meta_unique):,}")
logging.info(f" - source_meta_lookup rows indexed: {len(source_meta_lookup):,}")
logging.info(f"Done precomputing source metadata lookup | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

logging.info(f"Precomputing source compound sets | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

source_cpds = (sources_masked.groupby("source_id")["inchikey"].agg(set).to_dict())

logging.info(f" - Sources indexed: {len(source_cpds):,}")
logging.info(f"Done precomputing source compound sets | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()

logging.info(f"Preparing lookup tables for final enrichment | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# source_id → inchikey
source_compounds = sources_masked[["source_id", "inchikey"]]

logging.info(f"Done precomputing source_compounds | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# inchikey → feature_name (from sample dataset, e.g., dust)
dust_features = dust_full_std[["inchikey", "Feature Name"]].rename(
    columns={"Feature Name": "feature_name"}
).drop_duplicates()

logging.info(f"Done precomputing dust_features | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# source_id → source_name (all sources)
source_names = source_meta_unique[[
        "source_id",
        "source_name",
        "source_origin"
]]

logging.info(f"Done precomputing source_names | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# source_id → source kind metadata (all sources)
source_lookup = source_meta_unique[[
        "source_id",
        "source_kind",
        "source_general_category",
        "source_harmonized",
        "source_class",
        "source_class_domain"
]]

logging.info(f"Done precomputing source_lookup | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

minorsep()
logging.info(f"Source lookup tables ready | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# OPTIONAL: Export all source categories per origin (to redesign priors if needed)
majorsep()
logging.info(f"Exporting source categories per origin | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# One row per source_origin × source_kind × source_general_category, 
# with the number of sources and up to 3 example source names
# Upload the saved CSV, so priors can be assigned
category_export = (source_meta_unique
    .assign(
        source_kind=source_meta_unique["source_kind"].fillna("Unknown"),
        source_general_category=source_meta_unique["source_general_category"].fillna("Unknown"))
    .groupby(["source_origin", "source_kind", "source_general_category"])
    .agg(
        n_sources=("source_name", "size"),
        example_source_names=("source_name", lambda x: " | ".join(x.dropna().astype(str).unique()[:3])))
    .reset_index()
    .sort_values(["source_origin", "source_kind", "n_sources"], ascending=[True, True, False]))

logging.info(f"Rows (unique categories): {len(category_export):,}\n")
logging.info(category_export.groupby("source_origin").size().rename("n_unique_kind_category_pairs"))

LOOKUP_DIR = Path(DATA_DIR / "lookups")
LOOKUP_DIR.mkdir(exist_ok=True)
category_export.to_csv(LOOKUP_DIR / "source_categories_per_origin.csv", index=False)
minorsep()
logging.info(f"Saved source_categories_per_origin.csv to: \n{LOOKUP_DIR}")

minorsep()
logging.info(f"Done exporting source categories per origin | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Compound → source inverted index
majorsep()
logging.info(f"Building compound → source inverted index | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

compound_to_sources = defaultdict(list)

for sid, cpds in source_cpds.items():
    for ik in cpds:
        compound_to_sources[ik].append(sid)

logging.info(f" - Indexed compounds: {len(compound_to_sources):,}")
minorsep()
logging.info(f"Done building compound → source inverted index | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Define source weights & priors
# Every source_kind / source_general_category occurring in the source database
# (lookups/source_categories_per_origin.csv) has a value specified in this cell based on indoor dust data
# Should be adjusted for other matrices

# Scale: 1.0 = source type that is a well-documented, direct contributor to residential indoor dust
#        ~0.5 = plausible/indirect contributor
#        <=0.1 = mainly industrial/occupational or remote from homes
# Lookup per source (see "Compute household-independent Bayesian priors" cell):
#        kind_prior     = KIND table of its origin (hierarchical, default per origin if missing)
#        category_prior = CATEGORY table of its origin; if the category is not listed → category_prior = kind_prior
#        ontology_prior = sqrt(kind_prior * category_prior);   final prior = SOURCE_WEIGHT[origin] * ontology_prior

# Main evidence used for the relative values for dust:
# - Mitro et al. 2016, Environ Sci Technol 50(19):10661-10672 (meta-analysis of consumer-product chemicals in house dust:
#   phthalates > phenols > replacement flame retardants > fragrances > PFAS; sources: vinyl flooring, personal care,
#   cleaning products, building materials, furnishings)
# - Weschler & Nazaroff 2008, Atmos Environ 42(40):9018-9040 (SVOC partitioning from products/materials into indoor dust)
# - Lucattini et al. 2018, Chemosphere 201:466-482 (review: SVOCs in consumer products, indoor air and dust)
# - Layton & Beamer 2009, Environ Sci Technol 43(21):8199–8205 (track-in of outdoor soil/road dust into indoor dust)
# - Indoor combustion (wood stoves, cooking, tobacco smoke) and indoor pesticide/pet-treatment use are
#   well-known dust sources of PAHs, cooking-derived compounds, nicotine and pesticides

# Databases:
# - FooDB: https://foodb.ca/
# - SPECIEUROPE: https://source-apportionment.jrc.ec.europa.eu/Specieurope/index.aspx
# - SPECIATE: https://www.epa.gov/air-emissions-modeling/speciate
# - CPDat: https://epa.figshare.com/articles/dataset/The_Chemical_and_Products_Database_CPDat_MySQL_Data_File/5352997?file=53538266

# Literature consumer products references:
# - Park et al. 2024, Environ Eng Res 29(1):230123
# - Yun et al. 2023, Ecotoxicol Environ Saf 268:115695
# - Phillips et al. 2018, Environ Sci Technol 52:3125-3135
# - Lowe et al. 2021, Environ Sci Technol 55:11375-11387
# - Stanfield et al. 2025, Environ Sci Technol 59:1354-1366
# - Sørensen et al. 2023, J Haz Mat 458:1318107
# - Wu et al. 2023, J Chrom A 1711:464442


SOURCE_WEIGHT = {
    "HOUSEHOLD_PRODUCTS": 1.0,     # own analytical data (see HH relation factor in "Source prior helper")
    "LIT_CONSUMER_PRODUCTS": 0.8,  # analytical data from literature
    "FOODB": 0.7,                  # measured food composition
    "SPECIEUROPE": 0.6,            # measured European source profiles
    "SPECIATE": 0.4,               # measured US source profiles
    "CPDAT": 0.2                   # manufacturer-declared ingredients etc. (not measured)
}

# ---------------------------------------------------------------------------------------------
# Consumer products: LIT_CONSUMER_PRODUCTS & HOUSEHOLD_PRODUCTS share the same vocabulary
# ---------------------------------------------------------------------------------------------
CONS_PRODS_KIND_PRIOR = {
    "Fabric": 0.9,                   # textiles/upholstery: flame retardants, PFAS, dyes
    "Furniture": 0.9,                # foam/coatings: flame retardants, plasticizers
    "Construction Materials": 0.9,   # flooring, wall materials: phthalates, flame retardants
    "Electronics": 0.9,              # OPFRs/BFRs from casings & circuit boards
    "Cleaning Product": 0.9,         # direct indoor application
    "Personal Care Products": 0.9,   # fragrances, parabens, UV filters, siloxanes
    "Consumer Product": 0.8,
    "Shampoo": 0.8,
    "Children's Products": 0.8,      # toys, mats: plasticizers, flame retardants
    "Solvent-based": 0.8,            # sprays/coatings used indoors
    "Recycled": 0.8,                 # recycled plastics: legacy additives
    "Clothing": 0.7,
    "Plastic Product": 0.7,
    "Water-based": 0.6,
    "Virgin": 0.6,
    "Rubber Product": 0.6,
    "Kitchen": 0.4,
    "Unknown": 0.4
}

CONS_PRODS_CATEGORY_PRIOR = {
    # strongest indoor dust sources
    "Fabric Upholstery": 1.0, "Carpet": 1.0, "Floor": 0.9, "Construction Materials": 0.9,
    "Fabric Products": 0.9, "Furniture": 0.9, "Electronics": 0.9, "Air Freshener": 0.9,
    "Indoor": 0.9, "Polyether": 0.8, # PU foam (mattress): flame retardants
    "Formulation": 0.8, "Multipurpose": 0.8, "Children's Products": 0.8, "Shampoo": 0.8, "Fabric": 0.8,
    # moderate
    "Article": 0.7, "Plastic Home/auto Products": 0.7, "Plastic Home/Auto Products": 0.7,
    "Bathroom": 0.7, "Cotton Clothing": 0.7, "Baby Soap": 0.7, "Mosquito": 0.7, "Acari": 0.7,
    "Wrinkle Guard": 0.7, "Recycled Tire Products": 0.6, "SBR": 0.6, "SBR & PVC & NR": 0.6,
    "Polish": 0.6, "Kitchen": 0.5, "Window": 0.5, "Air Conditioner": 0.5,
    "PA": 0.5, "PET": 0.5, "EVA": 0.5, "EVS": 0.5, "SV": 0.5, "Elastomer": 0.5,
    # lower
    "PE": 0.4, "PP": 0.4, "PS": 0.4, "NR": 0.4, "Nitrile Rubber": 0.4, "Unsat Polyester": 0.4,
    "Silicone Kitchen Tools": 0.4, "Food Contact Materials": 0.4, "Rust Inhibitor": 0.4,
    "Paper Products": 0.3, "Glass": 0.3, "Metal": 0.3, "Car": 0.3, "PVA": 0.3, "Unknown": 0.3,
    "SBR & NR": 0.3, # end-of-life car tyres (outdoor)
    "Automobile": 0.2, "Food": 0.2, "Bowling Ball": 0.1
}

# ---------------------------------------------------------------------------------------------
# CPDat (product use category (PUC) kind = source_kind, PUC general category = source_general_category)
# ---------------------------------------------------------------------------------------------
PUC_KIND_PRIOR = {
    "Formulation": 1.0,   # products applied/used in homes
    "Article": 0.8,       # durable goods emitting SVOCs
    "NoID": 0.4,          # consumer products without PUC assignment (mostly personal care/household)
    "Unknown": 0.2,
    "Occupation": 0.1     # occupational/industrial use
}

PUC_CATEGORY_PRIOR = {
    # high
    "Furniture & Furnishings": 1.0, "Personal Care": 1.0, "Cleaning - Household": 0.9,
    "Construction Materials": 0.8, "Toys & Children Products": 0.8,
    "Home Maintenance": 0.7, "Pesticides": 0.7, # paints/adhesives; indoor biocides
    # moderate
    "Animal Care": 0.6, "Other Consumer Products": 0.6, # pet flea/tick treatments
    "Electronics": 0.5, "Arts & Office": 0.5, "Cleaning & Safety": 0.5,
    "Other Direct-Contact Consumer Goods": 0.5, "Food Contact Items": 0.5,
    "Food & Drugs": 0.4, "Machinery - Consumer": 0.4, "Packaging": 0.4,
    "Other Indirect-Contact Consumer Goods": 0.4, "Landscape & Yard": 0.4, "NoID": 0.4,
    # low
    "Construction": 0.3, "Sports": 0.3, "Tools": 0.3,
    "Batteries": 0.2, "Vehicles": 0.2, "Vehicles - Road": 0.2, "Medical": 0.2, "Unknown": 0.2,
    "Special": 0.1, "Other vehicles/mass transit": 0.1,
    "Manufacturing": 0.05, "Machinery - Industrial": 0.05, "Raw Materials": 0.05, "Industrial": 0.05,
    "Laboratory": 0.01
}

# ---------------------------------------------------------------------------------------------
# FooDB (source_kind = food group, source_general_category = food subgroup)
# Food compounds reach dust via food debris/crumbs & cooking emissions; most are low-to-moderate
# ---------------------------------------------------------------------------------------------
FOODB_KIND_PRIOR = {
    "Fats and oils": 0.5, "Herbs and Spices": 0.5, "Herbs and spices": 0.5, # cooking fumes, spice dust
    "Coffee and coffee products": 0.4, "Teas": 0.4,
    "Cereals and cereal products": 0.3, "Animal foods": 0.3, "Baking goods": 0.3, "Dishes": 0.3,
    "Snack foods": 0.3, "Cocoa and cocoa products": 0.3,
    "Aquatic foods": 0.2, "Baby foods": 0.2, "Beverages": 0.2, "Confectioneries": 0.2, "Eggs": 0.2,
    "Fruits": 0.2, "Milk and milk products": 0.2, "Nuts": 0.2, "Pulses": 0.2, "Soy": 0.2, "Vegetables": 0.2,
    "Gourds": 0.1, "Unclassified": 0.1
}

FOODB_CATEGORY_PRIOR = {
    # only subgroups that differ from their food group; all others use the food-group value
    "Spices": 0.6, "Herb and spice mixtures": 0.6, "Oilseed crops": 0.5, "Coffee": 0.5,
    "Vegetable fats": 0.5, "Animal fats": 0.4, "Fruit vegetables": 0.4, # incl. olive oil
    "Fish products": 0.3, "Tubers": 0.3, # fish oil; frying potatoes
    "Nutritional beverages": 0.1, "Waters": 0.05,
    # foods rarely present in European homes
    "Pinnipeds": 0.05, "Cetaceans": 0.05, "Amphibians": 0.05, "Other aquatic foods": 0.05,
    "Venison": 0.1, "Lagomorphs": 0.1, "Caprae": 0.1, "Ovis": 0.1
}

# ---------------------------------------------------------------------------------------------
# SPECIATE & SPECIEUROPE (same ontology; categories are hierarchical "A; B; C" → longest listed prefix is used)
# ---------------------------------------------------------------------------------------------
SPECIATE_KIND_PRIOR = {
    "Dust": 1.0,              # resuspended road/soil dust is tracked in and becomes part of house dust
    "Combustion": 0.8,        # PAHs etc. from traffic, stoves, cooking, smoking
    "Volatilization": 0.6,
    "Atomization": 0.6,
    "Microbial": 0.3,
    "Chemical Reaction": 0.1,
    "Miscellaneous": 0.3
}

SPECIATE_CATEGORY_PRIOR = {
    # indoor & near-home sources
    "Cooking": 1.0, "Cigarette": 1.0, "Consumer Products": 1.0, "Volatile Chemical Products": 1.0,
    "Misc. Non-industrial: Consumer and Commercial": 1.0,
    "Biomass Burning; Residential Wood Combustion": 0.9, "Residential": 0.8,
    "Surface Coating": 0.8, "Surface Coatings": 0.8, "Surface Coating; Architectural": 0.9,
    "Surface Coating; Wood Furniture": 0.8, "Surface Coating; Aerosols": 0.7, "Surface Coatings; Aerosol": 0.8,
    "Surface Coating; Flat Wood": 0.6, "Surface Coating; Metal Furniture": 0.6,
    "Surface Coating; Appliances": 0.6, "Surface Coating; Electrical": 0.5,
    "Pesticides": 0.8, "Dry Cleaning": 0.6, "Consumer Products; Degreasing": 0.6, "Solvent": 0.5,
    "Residential; Boiler": 0.5,
    # traffic & outdoor dust (track-in)
    "Road": 0.8, "Urban": 0.7, "Urban; Road": 0.8,
    "Mobile": 0.6, "Mobile; Onroad": 0.7, "Mobile; Onroad; Heavy Duty": 0.6, "Mobile; Moped": 0.5,
    "Mobile; Nonroad": 0.4, "Mobile; Aircraft": 0.3, "Mobile; Airport": 0.3, "Mobile; Marine": 0.2,
    "Geological": 0.4, "Mineral": 0.4, "Construction": 0.4, "Agriculture": 0.4,
    "Agriculture; Animal": 0.2, "Agriculture; Silage": 0.1, "Agriculture; Food": 0.2,
    "Biomass Burning": 0.6, "Biomass Burning; Biofuel": 0.5, "Biomass Burning; Open Burning": 0.4,
    "Biomass Burning; Wildfire": 0.4, "Biomass Burning; Prescribed Fire": 0.3, "Biomass Burning; Agriculture": 0.3,
    "Open Burning": 0.3, "Transport": 0.3, "Mineral Products": 0.2, "Composting": 0.2,
    "Biogenic Emissions": 0.3, "Secondary Aerosol": 0.2, "Secondary Organic Aerosol": 0.2,
    "Marine": 0.2, "Miscellaneous": 0.2,
    # industrial coatings & processes
    "Surface Coating; Spraying": 0.3, "Surface Coating; Traffic Paints": 0.3, "Surface Coating; Paper": 0.3,
    "Surface Coating; Adhesives": 0.4, "Surface Coating; Autos": 0.2, "Surface Coating; Industrial Maintenance": 0.2,
    "Surface Coating; Can": 0.2, "Surface Coating; Machinery": 0.2, "Surface Coating; Marine Paints": 0.2,
    "Surface Coating; Oil": 0.2, "Surface Coating; Aircraft": 0.1, "Surface Coating; Rail": 0.1,
    "Degreasing": 0.3, "Graphic Arts": 0.2, "Rubber And Plastics": 0.2, "Textile Products": 0.2,
    "Industrial; Surface Coating": 0.2, "Boiler": 0.2, "Engine": 0.2,
    "Industrial": 0.1, "Chemical Manufacturing": 0.1, "Paint Manufacturing": 0.1, "Incinerator": 0.1,
    "Landfill": 0.1, "Wastewater Treatment": 0.1, "Warehousing": 0.1, "Geogenic": 0.1,
    "Petrochemical": 0.05, "Refinery": 0.05, "Oil And Gas": 0.05, "Oil and gas": 0.05, "Pulp And Paper": 0.05,
    "Electric Generation": 0.05, "Turbine": 0.05, "Tank": 0.05, "Industrial; Port": 0.05,
    "Coke Oven": 0.02, "Basic Oxygen Furnace": 0.02, "Flare": 0.02
}

# One table per origin → (kind table, category table, default kind prior)
PRIOR_TABLES = {
    "HOUSEHOLD_PRODUCTS": (CONS_PRODS_KIND_PRIOR, CONS_PRODS_CATEGORY_PRIOR, 0.5),
    "LIT_CONSUMER_PRODUCTS": (CONS_PRODS_KIND_PRIOR, CONS_PRODS_CATEGORY_PRIOR, 0.5),
    "CPDAT": (PUC_KIND_PRIOR, PUC_CATEGORY_PRIOR, 0.2),
    "FOODB": (FOODB_KIND_PRIOR, FOODB_CATEGORY_PRIOR, 0.2),
    "SPECIATE": (SPECIATE_KIND_PRIOR, SPECIATE_CATEGORY_PRIOR, 0.3),
    "SPECIEUROPE": (SPECIATE_KIND_PRIOR, SPECIATE_CATEGORY_PRIOR, 0.3),
}

# Hierarchical prior lookup helper
def hierarchical_prior_lookup(category, prior_dict, default=0.3):
    if pd.isna(category):
        return default

    category = str(category).strip()

    # exact match first
    if category in prior_dict:
        return prior_dict[category]

    # hierarchical fallback ("A; B; C" → "A; B" → "A")
    parts = [p.strip() for p in category.split(";")]

    for i in range(len(parts), 0, -1):
        candidate = "; ".join(parts[:i])
        if candidate in prior_dict:
            return prior_dict[candidate]
    return default

majorsep()
logging.info(f"Source weights & priors defined | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Geo metadata influence weights
majorsep()
logging.info(f"Specifying geo metadata modules influence weights | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# 29 directional modules from geo_metadata_cleanup
MODULE_WEIGHT = {
    # indoor combustion
    "wood_burning_likelihood": 1.0,
    "gas_oil_combustion_likelihood": 0.8,
    "smoking_likelihood": 1.0,
    "vaping_likelihood": 0.0, # no e-cig/e-liquid source in the source database → set > 0 once one is added
    # indoor products & activities
    "fresheners_candles_likelihood": 0.9,
    "pesticides_household_likelihood": 0.9,
    "cleaning_products_likelihood": 0.7,
    "pets_likelihood": 0.7,
    "children_likelihood": 0.6,
    "occupancy_likelihood": 0.3,
    "electronics_likelihood": 0.6,
    # building & materials
    "floor_textile_likelihood": 0.6,
    "floor_resilient_likelihood": 0.6,
    "wood_surfaces_likelihood": 0.4,
    "building_age_likelihood": 0.5,
    "renovation_likelihood": 0.7,
    "ventilation_likelihood": 0.0, # more air exchange = more outdoor & less indoor → not directional, not used
    # outdoor
    "airport_likelihood": 0.8,
    "traffic_likelihood": 1.0,
    "industry_likelihood": 0.8,
    "agriculture_likelihood": 0.6,
    "agri_pesticides_likelihood": 0.6,
    "pfas_likelihood": 0.0, # no PFAS-specific source category in the source database → not used
    "urban_likelihood": 0.4,
    "green_likelihood": 0.4,
    "marine_likelihood": 0.4,
    "waste_likelihood": 0.4,
    "commercial_likelihood": 0.5,
    "outdoor_burning_likelihood": 0.5,
}

logging.info(f"Geo metadata module weights:\n{pd.Series(MODULE_WEIGHT)}")
minorsep()
logging.info(f"Specified geo metadata modules influence weights | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Map geo metadata modules to sources
majorsep()
logging.info(f"Mapping geo metadata modules to sources | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Uses 29 directional geo modules (household_metadata_priors.csv from geo_metadata_cleanup)
# Every label below exists in the source database (checked against lookups/source_categories_per_origin.csv)

# Principles:
#   - a source category is linked only to modules whose variables describe that source directly
#   - most categories are linked to one module, at most two, so no source gets a boost from overlapping evidence
#   - modules with MODULE_WEIGHT = 0 (vaping, ventilation, pfas) have no links

# Matching rules (geo_label_matches below):
#   - a label matches a source if it equals its source_general_category or its source_kind
#   - a label ending with "*" also matches all sub-categories ("Mobile; Onroad*" → light & heavy duty too)
#   - labels are matched per source_origin only, e.g., LIT_CONSUMER_PRODUCTS "Floor" = floor cleaner,
#     HOUSEHOLD_PRODUCTS "Floor" = flooring material

# A linked source gets its prior multiplied by exp(MODULE_WEIGHT * (household likelihood - 0.5))

MODULE_SOURCE_MAP = {
    # indoor combustion
    "wood_burning_likelihood": {
        "SPECIATE": ["Biomass Burning; Residential Wood Combustion*", "Biomass Burning; Biofuel"],
        "SPECIEUROPE": ["Biomass Burning; Residential Wood Combustion*", "Biomass Burning; Biofuel"]
    },
    "gas_oil_combustion_likelihood": {
        "SPECIATE": ["Residential*"], # residential gas/oil combustion & boilers
        "SPECIEUROPE": ["Residential*"]
    },
    "smoking_likelihood": {
        "SPECIATE": ["Cigarette"]
    },
    "vaping_likelihood": {}, # weight 0: no e-cig source in the source database

    # indoor products & activities
    "fresheners_candles_likelihood": {
        "LIT_CONSUMER_PRODUCTS": ["Formulation", "Indoor"], # air fresheners 1-60, indoor deodorizers
        "HOUSEHOLD_PRODUCTS": ["Air Freshener"] # (scented) candles
    },
    "pesticides_household_likelihood": {
        "CPDAT": ["Pesticides"], # consumer biocides/insecticides
        "LIT_CONSUMER_PRODUCTS": ["Mosquito", "Acari"] # mosquito & mite products
    },
    "cleaning_products_likelihood": {
        "CPDAT": ["Cleaning - Household"],
        "LIT_CONSUMER_PRODUCTS": ["Cleaning Product", # kind: all cleaners
                                  "Bathroom", "Window", "Multipurpose", "Floor"], # incl. solvent/water-based cleaners
        "SPECIATE": ["Consumer Products*"], # volatile consumer-product profiles (mostly cleaners/aerosols)
        "SPECIEUROPE": ["Consumer Products"]
    },
    "pets_likelihood": {
        "CPDAT": ["Animal Care"] # flea/tick treatments, pet shampoos
    },
    "children_likelihood": {
        "CPDAT": ["Toys & Children Products"],
        "LIT_CONSUMER_PRODUCTS": ["Children's Products", "Baby Soap"],
        "HOUSEHOLD_PRODUCTS": ["Children's Products"],
        "FOODB": ["Baby foods"]
    },
    "occupancy_likelihood": { # more occupants → more personal care, cooking, food handling
        "CPDAT": ["Personal Care"],
        "LIT_CONSUMER_PRODUCTS": ["Personal Care Products", "Shampoo"],
        "SPECIATE": ["Cooking*"],
        "FOODB": ["Cereals and cereal products", "Baking goods", "Fats and oils", "Herbs and Spices", "Herbs and spices",
                  "Coffee and coffee products", "Teas", "Milk and milk products", "Animal foods", "Dishes",
                  "Snack foods", "Cocoa and cocoa products", "Confectioneries"] # food groups (FooDB source_kind)
    },
    "electronics_likelihood": {
        "CPDAT": ["Electronics", "Machinery - Consumer", "Batteries"],
        "LIT_CONSUMER_PRODUCTS": ["Air Conditioner"],
        "HOUSEHOLD_PRODUCTS": ["Electronics"]
    },

    # building & materials
    "floor_textile_likelihood": {
        "LIT_CONSUMER_PRODUCTS": ["Carpet", "Recycled Tire Products"], # carpets; rubber rugs/mats
        "HOUSEHOLD_PRODUCTS": ["Carpet"],
        "CPDAT": ["Furniture & Furnishings"] # incl. carpets & rugs
    },
    "floor_resilient_likelihood": {
        "LIT_CONSUMER_PRODUCTS": ["SV", "EVA", "EVS"], # vinyl flooring
        "HOUSEHOLD_PRODUCTS": ["Floor"], # flooring (Construction Materials)
        "CPDAT": ["Construction Materials"]
    },
    "wood_surfaces_likelihood": {
        "SPECIATE": ["Surface Coating; Wood Furniture", "Surface Coating; Flat Wood"],
        "LIT_CONSUMER_PRODUCTS": ["Polish"], # polish coating agent
        "CPDAT": ["Home Maintenance"], # varnishes, stains, sealants
        "HOUSEHOLD_PRODUCTS": ["Furniture"]
    },
    "building_age_likelihood": { # assumes higher = older building
        "LIT_CONSUMER_PRODUCTS": ["Construction Materials"],
        "HOUSEHOLD_PRODUCTS": ["Construction Materials"],
        "SPECIATE": ["Surface Coating; Architectural"] # old paints
    },
    "renovation_likelihood": {
        "CPDAT": ["Construction", "Home Maintenance", "Construction Materials"],
        "LIT_CONSUMER_PRODUCTS": ["SV", "EVA", "EVS"], # new vinyl flooring
        "SPECIATE": ["Surface Coating; Architectural", "Solvent"] # fresh paints, solvents
    },
    "ventilation_likelihood": {}, # weight 0: not directional

    # outdoor
    "airport_likelihood": {
        "SPECIATE": ["Mobile; Aircraft", "Mobile; Airport", "Surface Coating; Aircraft"]
    },
    "traffic_likelihood": {
        "SPECIATE": ["Mobile", "Mobile; Onroad*", "Mobile; Moped", "Road", "Transport",
                     "Surface Coating; Traffic Paints"], # "Mobile"/"Transport" = fuels, refuelling
        "SPECIEUROPE": ["Mobile; Onroad*", "Road", "Urban; Road"],
        "CPDAT": ["Vehicles", "Vehicles - Road"], # fuels/fluids, brake linings
        "LIT_CONSUMER_PRODUCTS": ["SBR & NR"] # tyre rubber
    },
    "industry_likelihood": {
        "SPECIATE": ["Industrial*", "Chemical Manufacturing*", "Petrochemical*", "Refinery*", "Oil And Gas*",
                     "Oil and gas*", "Pulp And Paper*", "Electric Generation*", "Mineral Products*", "Coke Oven",
                     "Basic Oxygen Furnace", "Boiler", "Engine", "Turbine", "Flare", "Tank", "Warehousing",
                     "Paint Manufacturing", "Rubber And Plastics", "Textile Products", "Degreasing", "Construction"],
        "SPECIEUROPE": ["Industrial", "Industrial; Boiler", "Industrial; Metal*", "Industrial; Furnace", "Coke Oven",
                        "Electric Generation*", "Boiler", "Construction"], # "Industrial; Marine/Port" → marine
        "CPDAT": ["Industrial", "Manufacturing", "Raw Materials", "Machinery - Industrial"]
    },
    "agriculture_likelihood": { # farms & crops around the household
        "SPECIATE": ["Agriculture*"],
        "FOODB": ["Cereals and cereal products", # wheat, barley, rye, oats, maize, rice, other cereals
                  "Tubers", # potatoes
                  "Root vegetables", "Sugars", # sugar beet, other root crops
                  "Oilseed crops", "Vegetable fats", # sunflower, rape
                  "Soy", "Pulses", # soya, dry pulses
                  "Fruits", "Nuts", "Fruit vegetables"] # permanent crops (orchards, olives, vineyards)
    },
    "agri_pesticides_likelihood": { # modelled agricultural pesticide use around the household
        "SPECIATE": ["Pesticides"],
        "CPDAT": ["Pesticides"] # only the source database source of many pesticide active ingredients
    },
    "pfas_likelihood": {}, # weight 0: no PFAS-specific source category in the source database
    "urban_likelihood": {
        "SPECIATE": ["Urban"],
        "SPECIEUROPE": ["Urban"] # "Urban; Road" is linked to traffic
    },
    "green_likelihood": {
        "SPECIEUROPE": ["Biogenic Emissions", "Mineral"], # vegetation; soil/mineral dust
        "SPECIATE": ["Geological"], # soil
        "CPDAT": ["Landscape & Yard"] # garden products
    },
    "marine_likelihood": {
        "SPECIEUROPE": ["Marine", "Mobile; Marine*", "Industrial; Marine", "Industrial; Port"],
        "SPECIATE": ["Mobile; Marine", "Surface Coating; Marine Paints"]
    },
    "waste_likelihood": {
        "SPECIATE": ["Landfill", "Wastewater Treatment", "Composting", "Incinerator"]
    },
    "commercial_likelihood": { # coffee bars/restaurants, laundries, print shops
        "SPECIATE": ["Dry Cleaning", "Graphic Arts", "Misc. Non-industrial: Consumer and Commercial", "Cooking*"]
    },
    "outdoor_burning_likelihood": {
        "SPECIATE": ["Open Burning", "Biomass Burning; Wildfire*", "Biomass Burning; Prescribed Fire",
                     "Biomass Burning; Agriculture"],
        "SPECIEUROPE": ["Biomass Burning; Open Burning"]
    }
}

# Matching rule used by metadata_multiplier() and by the geo check cell
def geo_label_matches(label, kind, category):
    kind = "" if pd.isna(kind) else str(kind).strip()
    category = "" if pd.isna(category) else str(category).strip()

    if label.endswith("*"):
        base = label[:-1].strip()
        return (category == base or category.startswith(base + ";")
                or kind == base or kind.startswith(base + ";"))

    return label == category or label == kind

# If this cell is re-run after the helper cells, forget the cached module links of the old map
if "source_geo_modules" in globals():
    source_geo_modules.cache_clear()

# Consistency checks between MODULE_WEIGHT, MODULE_SOURCE_MAP and the geo metadata file
file_modules = set(module_cols) if USE_GEO_PRIORS else set(MODULE_WEIGHT) # no file → skip the file-vs-weights comparison

weight_modules = set(MODULE_WEIGHT)
map_modules = set(MODULE_SOURCE_MAP)

checks = {
    "In geo file but NOT in MODULE_WEIGHT (ignored)": sorted(file_modules - weight_modules),
    "In MODULE_WEIGHT but NOT in geo file (no effect)": sorted(weight_modules - file_modules),
    "In MODULE_WEIGHT but NOT in MODULE_SOURCE_MAP": sorted(weight_modules - map_modules),
    "In MODULE_SOURCE_MAP but NOT in MODULE_WEIGHT": sorted(map_modules - weight_modules),
    "Weight > 0 but no linked sources": sorted(m for m in weight_modules if MODULE_WEIGHT[m] > 0 and not MODULE_SOURCE_MAP.get(m)),
    "Linked sources but weight 0": sorted(m for m in map_modules if MODULE_SOURCE_MAP[m] and MODULE_WEIGHT.get(m, 0) == 0),
}

for message, modules in checks.items():
    if modules:
        logging.warning(f" - {message}: {modules}")

logging.info(f" - {len(MODULE_SOURCE_MAP)} metadata modules mapped | {sum(1 for m in MODULE_SOURCE_MAP.values() if m)} with linked sources")
minorsep()
logging.info(f"Mapped geo metadata modules to sources | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Check geo metadata mapping (which MODULE_SOURCE_MAP labels actually match the source database)
majorsep()
logging.info(f"Checking geo metadata mapping against the source database | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

geo_check_rows = []

for module, db_map in MODULE_SOURCE_MAP.items():
    for db, labels in db_map.items():
        db_meta = source_meta_unique[source_meta_unique["source_origin"] == db]

        for label in labels:
            n_matched = int(sum(geo_label_matches(label, k, c)
                                for k, c in zip(db_meta["source_kind"], db_meta["source_general_category"])))

            if len(db_meta) == 0:
                status = "ORIGIN NOT IN DB"
            elif n_matched > 0:
                status = "OK"
            else:
                status = "NOT FOUND"

            geo_check_rows.append({
                "module": module,
                "source_origin": db,
                "label": label,
                "status": status,
                "n_sources_matched": n_matched
            })

geo_check_df = pd.DataFrame(geo_check_rows)

logging.info(f"Label status (count):")
logging.info(geo_check_df["status"].value_counts())
minorsep()
logging.info(f"Labels that do NOT fire (should be empty):")
logging.info(geo_check_df[geo_check_df["status"] != "OK"])
minorsep()

# Which part of each source origin can be reached by geo priors at all?
reach_rows = []
for origin, grp in source_meta_unique.groupby("source_origin"):
    labels = [lab for db_map in MODULE_SOURCE_MAP.values() for lab in db_map.get(origin, [])]
    n_reached = int(sum(any(geo_label_matches(lab, k, c) for lab in labels)
                        for k, c in zip(grp["source_kind"], grp["source_general_category"])))
    reach_rows.append({"source_origin": origin, "n_sources": len(grp), "n_sources_reached_by_geo_priors": n_reached,
                       "share_reached": round(n_reached / len(grp), 3) if len(grp) else 0})
geo_reach_df = pd.DataFrame(reach_rows)
logging.info(f"Share of sources per origin that geo priors can influence:")
logging.info(geo_reach_df)
minorsep()

# Do the sample IDs match the household IDs in the geo file? (match on LLNN)
sample_ids = [c for c in sample_universes_df.columns if c != "inchikey"]
geo_ids = list(hh_metadata_lookup.keys())
sample_keys = {s: household_key(s) for s in sample_ids}
matched = sorted(s for s, k in sample_keys.items() if k in hh_metadata_lookup)
logging.info(f" - Samples matched to geo metadata: {len(matched):,} / {len(sample_ids):,}" + ("" if USE_GEO_PRIORS else " (USE_GEO_PRIORS = False → geo priors not applied)"))
logging.info(f" - Samples without geo metadata: \n   {sorted(s for s, k in sample_keys.items() if k not in hh_metadata_lookup)}")
logging.info(f"\n - Geo households without a sample: \n   {sorted(set(map(str, geo_ids)) - set(sample_keys.values()))}")
minorsep()

geo_check_df.to_csv(GEO_DIR / "geo_mapping_check.csv", index=False)
logging.info(f"Saved geo_mapping_check.csv to: \n{GEO_DIR}")

minorsep()
logging.info(f"Done checking geo metadata mapping | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Geo metadata Bayesian helper functions
majorsep()
logging.info(f"Specifying geo metadata Bayesian helper functions | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

def household_metadata(household_id):
    return hh_metadata_lookup.get(household_key(household_id), {}) # 'FR01_LS2' → 'FR01'

# Which geo modules is a source linked to? Depends only on (origin, kind, category), 
# so it is computed once per label combination and cached
# Matching uses geo_label_matches() from the "Map geo metadata modules" cell
# (category OR kind; labels ending with "*" also match sub-categories)

@lru_cache(maxsize=None)
def source_geo_modules(database_name, kind, category):
    return tuple(module for module, db_map in MODULE_SOURCE_MAP.items()
                 if any(geo_label_matches(label, kind, category) for label in db_map.get(database_name, [])))

def metadata_multiplier(household_id, database_name, category, kind=None):
    meta = household_metadata(household_id)

    if not meta:
        return 1.0

    log_multiplier = 0.0

    for module in source_geo_modules(str(database_name), str(kind), str(category)): # each linked module counts once
        likelihood = meta.get(module, 0.5)
        weight = MODULE_WEIGHT.get(module, 0.5)
        log_multiplier += weight * (likelihood - 0.5)

    return float(np.exp(log_multiplier))

logging.info(f"Specified geo metadata Bayesian helper functions | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Compute household-independent Bayesian priors
majorsep()
logging.info(f"Precomputing Bayesian source priors | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

source_prior_lookup = {}

for sid, row in source_meta_lookup.iterrows():
    source_origin = row.get("source_origin", "Unknown")
    kind = row.get("source_kind", "Unknown")
    category = row.get("source_general_category", "Unknown")

    # Database confidence
    origin_weight = SOURCE_WEIGHT.get(source_origin, 0.1)

    # One lookup table per origin (PRIOR_TABLES in the "Define source weights & priors" cell)
    # The category prior falls back to the kind prior when the category is not listed
    kind_dict, category_dict, kind_default = PRIOR_TABLES.get(source_origin, ({}, {}, 0.1))
    kind_prior = hierarchical_prior_lookup(kind, kind_dict, default=kind_default)
    category_prior = hierarchical_prior_lookup(category, category_dict, default=None)

    if category_prior is None:
        category_prior = kind_prior
    
    # Geometric fusion of ontology priors
    ontology_prior = np.sqrt(kind_prior * category_prior)

    # Final household-independent prior
    final_prior = origin_weight * ontology_prior
    
    source_prior_lookup[sid] = float(final_prior)

logging.info(f" - Bayesian priors computed: {len(source_prior_lookup):,}")

minorsep()
logging.info(f"Done precomputing Bayesian source priors | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Source prior helper
majorsep()
logging.info(f"Defining source prior helper | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Precompute source_metadata_lookup once
# This avoids Pandas .loc lookup, which, during GREEDY, can happen millions of times
logging.info(f" - Precomputing source_metadata_lookup dictionary once | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

source_metadata_lookup_dict = {}

for sid, row in source_meta_lookup.iterrows():
    source_metadata_lookup_dict[sid] = {
        "origin": row["source_origin"],
        "kind": row["source_kind"],
        "category": row["source_general_category"]
    }

logging.info(f" - Precomputed source_metadata_lookup dictionary | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Household-aware priority of own analyzed household products (HOUSEHOLD_PRODUCTS)
# Product IDs look like 'FR01_CP04_HHP' (country FR, sample 01, product 04, HHP = source database suffix) → household 'FR01'
# The prior of such a product is multiplied by a factor depending on where it comes from relative to the sample
# With HH_OTHER_COUNTRY = 0.2 (= SOURCE_WEIGHT of CPDAT), an analyzed product from another country is about as
# credible as a manufacturer-declared CPDat product; products from the same household are strongly preferred
HH_SAME_HOUSEHOLD = 2.0 # to strengthen own HH product choice
HH_SAME_COUNTRY = 0.5
HH_OTHER_COUNTRY = 0.2

HH_PRODUCT_ID_PATTERN = re.compile(r"^([A-Z]{2})(\d{2})_CP\d+_HHP$")

def hh_product_household(source_id):
    m = HH_PRODUCT_ID_PATTERN.match(str(source_id).strip().upper())
    return f"{m.group(1)}{m.group(2)}" if m else None

def hh_relation_factor(source_id, household_id):
    product_hh = hh_product_household(source_id)
    sample_hh = household_key(household_id)

    if product_hh is None or sample_hh is None:
        return HH_OTHER_COUNTRY # unknown origin of the product → treated like another country
    if product_hh == sample_hh:
        return HH_SAME_HOUSEHOLD
    if product_hh[:2] == sample_hh[:2]:
        return HH_SAME_COUNTRY
    return HH_OTHER_COUNTRY

# Check that all household product IDs can be read
hh_ids = [sid for sid, m in source_metadata_lookup_dict.items() if m["origin"] == "HOUSEHOLD_PRODUCTS"]
hh_unreadable = [sid for sid in hh_ids if hh_product_household(sid) is None]
logging.info(f" - Household products: {len(hh_ids):,} | Household readable from ID: {len(hh_ids) - len(hh_unreadable):,}")
if hh_unreadable:
    logging.warning(f" - Household product IDs NOT in format LLNN_llnn_HHP (treated as other country): {hh_unreadable[:20]}")

def source_prior(source_id, household_id=None):
    base_prior = source_prior_lookup[source_id]

    if household_id is None:
        return base_prior

    meta = source_metadata_lookup_dict[source_id]

    multiplier = metadata_multiplier(
        household_id=household_id,
        database_name=meta["origin"],
        category=meta["category"],
        kind=meta["kind"] # geo labels can also match source_kind
    )

    # household-aware priority of own analyzed household products
    if meta["origin"] == "HOUSEHOLD_PRODUCTS":
        multiplier *= hh_relation_factor(source_id, household_id)

    return base_prior * multiplier

minorsep()
logging.info(f"Done defining source prior helper | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Save source metadata & source prior weights lookups & source compound sets
majorsep()
logging.info(f"Saving source metadata & source prior weights lookups & source compound sets | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

LOOKUP_DIR = Path(DATA_DIR / "lookups")
LOOKUP_DIR.mkdir(exist_ok=True)

source_meta_lookup.to_csv(LOOKUP_DIR / "source_metadata_lookup.csv", index=True)

source_metadata_lookup_dict_df = (pd.DataFrame(source_metadata_lookup_dict.items(), columns=["source_id", "origins"]))
source_metadata_lookup_dict_df.to_csv(LOOKUP_DIR / "source_metadata_lookup_dict.csv", index=False)

source_prior_lookup_df = (pd.DataFrame(source_prior_lookup.items(), columns=["source_id", "source_weight"]))
source_prior_lookup_df.to_csv(LOOKUP_DIR / "source_prior_lookup.csv", index=False)

source_cpds_df = (pd.DataFrame(source_cpds.items(), columns=["source_id", "inchikey"]))
source_cpds_df.to_csv(LOOKUP_DIR / "source_compound_sets.csv", index=False)

hh_metadata_lookup_df = (pd.DataFrame(hh_metadata_lookup.items(), columns=["household_id", "likelihoods"]))
hh_metadata_lookup_df.to_csv(LOOKUP_DIR / "hh_metadata_lookup.csv", index=False)

compound_to_sources_df = (pd.DataFrame(compound_to_sources.items(), columns=["inchikey", "sources"]))
compound_to_sources_df.to_csv(LOOKUP_DIR / "compound_to_sources_lookup.csv", index=False)

logging.info(f"Saved source metadata & source prior weights lookups & source compound sets | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Build optimization problem
majorsep()
logging.info(f"Defining optimization problem builder | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

#  - uncertainty_sigma default changed 0.25 → 0.0: the old lognormal "noise" was re-seeded with the same seed
#    on every call, so it was a fixed random distortion of the costs, not an uncertainty estimate
#  - seed is now passed through, so if you ever want real uncertainty, run the whole analysis for several seeds
def build_problem(compounds, household_id=None, alpha=0.0, uncertainty_sigma=0.0, seed=GLOBAL_SEED):
    compounds = set(compounds)
    A = []
    source_index = []

    for sid, cpds in source_cpds.items():
        elems = cpds & compounds

        if elems:
            A.append(elems)
            source_index.append(sid)

    if not A:
        return None, None, None

    P = make_costfun(
        A=A,
        source_index=source_index,
        source_meta_lookup=source_meta_lookup,
        household_id=household_id,
        seed=seed,
        # alpha_prior=2.0,
        alpha_prior=PRIOR_STRENGTH, # set in "Calibrate prior strength" cell (2.0 before that)
        beta_coverage=1.0,
        uncertainty_sigma=uncertainty_sigma,
    )

    assert len(A) == len(source_index) == len(P), "A, source_index, and P have inconsistent lengths"

    return A, source_index, P

logging.info(f"Done defining optimization problem builder | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Cost function
majorsep()
logging.info(f"Initializing probabilistic cost function | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

"""
Source probability model:
p(source_i) ∝ prior_i^alpha_prior × coverage_i^beta_coverage × uncertainty
prior_i      = metadata-derived prior
coverage_i   = fraction of compounds explained
uncertainty  = optional lognormal perturbation (OFF by default since 260929)
alpha_prior = 0 → ignore metadata completely
beta_coverage = 0 → use only metadata
both = 1 → multiplicative Bayesian-style model
alpha_prior > 1 → stronger preference for metadata-supported sources
beta_coverage > 1 → stronger preference for chemically comprehensive sources

Cost:
P_i = -log p(source_i)
"""

# Strength of the priors in the cost (exponent on the prior); 2.0 = old fixed value
# "Calibrate prior strength" cell (after alpha tuning) overwrites it
if "PRIOR_STRENGTH" not in globals():
    PRIOR_STRENGTH = 2.0

def make_costfun(
    A,
    source_index,
    source_meta_lookup,
    household_id=None,
    seed=GLOBAL_SEED,
    # alpha_prior=2.0,
    alpha_prior=None, # None → use PRIOR_STRENGTH
    beta_coverage=1.0,
    # uncertainty_sigma=0.25
    uncertainty_sigma=0.0 # noise OFF by default (see "Build optimization problem" cell)
):
    rng = np.random.default_rng(seed)
    
    if alpha_prior is None:
        alpha_prior = PRIOR_STRENGTH
        
    # Coverage contribution
    sizes = np.array([len(a) for a in A], dtype=float)

    if sizes.sum() > 0:
        sizes /= sizes.sum()
    else:
        sizes = np.full(len(A), 1 / len(A))
    
    # Clip potential zeros
    sizes = np.clip(sizes, 1e-12, None)

    # Household-aware Bayesian priors
    priors = np.array([source_prior(sid, household_id=household_id)
            for sid in source_index], dtype=float)

    # Avoid zero priors
    priors = np.clip(priors, 1e-6, None)

    # Same model, computed in log space: with a larger prior strength, priors ** alpha_prior becomes extremely small
    # and the old clip at 1e-12 would have made all low-prior sources equally (and wrongly) cheap
    log_probs = alpha_prior * np.log(priors) + beta_coverage * np.log(sizes)

    if uncertainty_sigma > 0: # identical to multiplying by a lognormal factor
        log_probs = log_probs + rng.normal(loc=0.0, scale=uncertainty_sigma, size=len(log_probs))

    log_probs = log_probs - log_probs.max()
    log_norm = np.log(np.exp(log_probs).sum())
    P = -(log_probs - log_norm) # = -log(normalized probability); always >= 0

    assert np.all(np.isfinite(P)), "Cost vector contains non-finite values"
    
    return P

logging.info(f"Done initializing cost function | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# GREEDY (Single Cover) + Shared Helper Functions
majorsep()
logging.info(f"Defining GREEDY (Single Cover) algorithm | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Precompute static source costs
def calculate_source_costs(source_index, U_set, P, alpha):
    source_cost = np.empty(len(source_index), dtype=float)

    for i, sid in enumerate(source_index):
        extras = source_cpds[sid] - U_set
        raw_penalty = (sum(compound_penalty.get(c, 0.0) for c in extras) / max(len(source_cpds[sid]), 1))
        penalty = np.clip((raw_penalty - PENALTY_MIN) / PENALTY_RANGE, 0.0, 1.0) # guarantees penalty stays in [0, 1]
        source_cost[i] = P[i] + alpha * penalty

        assert 0.0 <= penalty <= 1.0, "Normalized penalty outside [0,1]"
        
    return source_cost

# Calculate incremental cost of adding source i
def score_candidate(i, C, A, source_cost):
    R = set(A[i]) - C

    if not R:
        return None

    redundancy = len(R & C) # set(A[i]) - C already removed everything in C, so redundancy = 0
    w = (source_cost[i] + 0.25 * redundancy) / len(R)

    return w, R

# Update compound coverage cost vector
def update_coverage(e, R, w, U_index):
    e_new = e.copy()

    for elem in R:
        idx = U_index[elem]

        if np.isnan(e_new[idx]):
            e_new[idx] = w

    return e_new

# Helper to generate next tree branches
def build_children(x, C, A, source_cost, k, rng, replace):
    candidates = []

    for i in range(len(A)):
        if i in x:
            continue

        result = score_candidate(i, C, A, source_cost)

        if result is None:
            continue

        w, R = result
        candidates.append((i, w, R))

    if not candidates:
        return []

    candidates.sort(key=lambda t: t[1])

    top_k = candidates[:min(10, len(candidates))]

    weights = np.array([1 / (w + 1e-9) for _, w, _ in top_k])
    weights /= weights.sum()

    idx = rng.choice(len(top_k), size=min(k, len(top_k)), replace=replace, p=weights)

    return [top_k[i] for i in idx]

# GREEDY
def greedy_cover(A, U, P, source_index, alpha=0.0):
    C = set()
    x = set()
    e = np.full(len(U), np.nan)

    U_index = {u: i for i, u in enumerate(U)}
    U_set = set(U)

    source_cost = calculate_source_costs(source_index, U_set, P, alpha)

    while C != U_set:
        best_i = None
        best_w = np.inf
        best_R = None

        for i in range(len(A)):
            if i in x:
                continue

            result = score_candidate(i, C, A, source_cost)

            if result is None:
                continue

            w, R = result

            if w < best_w:
                best_w = w
                best_i = i
                best_R = R

        if best_i is None:
            break

        x.add(best_i)
        e = update_coverage(e, best_R, best_w, U_index)
        C |= best_R

    x_out = np.zeros(len(A))
    x_out[list(x)] = 1

    assert set().union(*[A[i] for i in np.where(x_out == 1)[0]]) >= U_set, "GREEDY failed to cover the universe"

    return x_out, e

logging.info(f"Done defining GREEDY (Single Cover) algorithm | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Beam node
majorsep()
logging.info(f"Defining beam node | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

@dataclass(slots=True)
class BeamNode:
    priority: float
    selected: tuple
    covered: frozenset
    remaining: frozenset
    coverage: np.ndarray
    cost: float
    entropy: float
    depth: int

logging.info(f"Done defining beam node | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Beam priority score
majorsep()
logging.info(f"Defining beam priority score | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

def beam_priority(cost, uncovered, depth, lambda_cover=6.0, lambda_depth=0.05):
    return cost + lambda_cover * uncovered + lambda_depth * depth

logging.info(f"Done defining beam priority score | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Beam Search multiple-cover algorithm
majorsep()
logging.info(f"Defining Beam Search cover algorithm | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

#  - Fix: previously each beam node only looked at the first 20 candidate sources in an arbitrary
#    (Python hash-dependent) order, so the search was essentially random and not reproducible between kernel restarts
#  - Now all candidate sources are scored and sorted deterministically
#  - If this becomes too slow, set BEAM_MAX_CANDIDATES below (e.g., 2000): only the cheapest candidate sources per node are then scored
BEAM_MAX_CANDIDATES = None # None = score all candidate sources (most correct, but slowest)

def beam_search_covers(
    A,
    U,
    P,
    source_index,
    alpha=0.0,
    beam_width=20,
    max_children=5,
    max_covers=50,
    diversity_lambda=0.15, # not used inside the function (kept for compatibility)
    seed=GLOBAL_SEED,
    max_candidates=BEAM_MAX_CANDIDATES
):
    rng = np.random.default_rng(seed)

    U = list(U)
    U_set = set(U)
    U_index = {u: i for i, u in enumerate(U)}

    source_lookup_beam = {sid: i for i, sid in enumerate(source_index)}
    source_cost = calculate_source_costs(source_index, U_set, P, alpha)

    beam = [
        BeamNode(
            priority=beam_priority(0.0, len(U_set), 0),
            selected=(),
            covered=frozenset(),
            remaining=frozenset(U_set),
            coverage=np.full(len(U), np.nan),
            cost=0.0,
            entropy=0.0,
            depth=0
        )
    ]

    finished = []

    progress = tqdm(total=max_covers, desc="Beam search", leave=False)

    while beam:
        next_beam = []
    
        for node in beam:
            if len(finished) >= max_covers:
                break
    
            if not node.remaining:
                finished.append(node)

                progress.update(1)

                if len(finished) >= max_covers:
                    break
                continue
    
            candidates = []
    
            # for i in source_order:
            candidate_sources = set()

            for ik in node.remaining:
                candidate_sources.update(compound_to_sources.get(ik, []))
            
            candidate_indices = [source_lookup_beam[sid] for sid in candidate_sources if sid in source_lookup_beam]

            # deterministic order (cheapest first, ties by index)
            candidate_indices.sort(key=lambda i: (source_cost[i], i))

            # optional speed cap, applied after sorting
            if max_candidates is not None:
                candidate_indices = candidate_indices[:max_candidates]
            
            for i in candidate_indices:
                if i in node.selected:
                    continue

                gain = len(node.remaining & A[i])
    
                if gain == 0:
                    continue

                score = gain / max(source_cost[i], 1e-12) # avoids division by zero
    
                candidates.append((score, i))
    
            if not candidates:
                continue
    
            # candidates.sort(reverse=True)
            candidates.sort(key=lambda t: (-t[0], t[1])) # best score first, deterministic ties
    
            for _, i in candidates[:max_children]:
                new_selected = node.selected + (i,) # less expensive; duplicates already prevent anyway
                new_covered = node.covered | A[i]
                new_remaining = frozenset(node.remaining - A[i])
    
                new_cov = update_coverage(
                    node.coverage.copy(),
                    A[i],
                    source_cost[i],
                    U_index
                )
    
                new_cost = node.cost + source_cost[i]
    
                priority = beam_priority(
                    new_cost,
                    len(new_remaining),
                    node.depth + 1
                )
    
                next_beam.append(
                    BeamNode(
                        priority=priority,
                        selected=new_selected,
                        covered=frozenset(new_covered),
                        remaining=new_remaining,
                        coverage=new_cov,
                        cost=new_cost,
                        entropy=0.0,
                        depth=node.depth + 1
                    )
                )
    
        if not next_beam:
            break

        next_beam.sort(key=lambda n: (n.priority, tuple(sorted(n.selected)))) # deterministic ties

        beam = []
        seen = set()
        
        for node in next_beam:
            key = (node.remaining, len(node.selected))
        
            if key in seen:
                continue
        
            seen.add(key)
            beam.append(node)
        
            if len(beam) >= beam_width:
                break

    progress.close()

    finished = sorted(finished, key=lambda x: (x.cost, tuple(sorted(x.selected))))[:max_covers] # deterministic ties

    covers = []
    seen = set()
    
    for node in finished:
        key = tuple(sorted(node.selected))

        if key in seen:
            continue

        seen.add(key)
        covers.append((list(node.selected), node.coverage.copy(), node.cost))
    
        if len(covers) >= max_covers:
            break
        
    for x_set, _, _ in covers:
        assert set().union(*[A[i] for i in x_set]) >= U_set, "Beam search produced incomplete cover"

    return covers

logging.info(f"Done defining Beam Search cover algorithm | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# OPTIONAL: Tiered set cover by source origin
majorsep()
logging.info(f"Defining tiered set cover (by SOURCE_WEIGHT) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# How it works:
# - Source origins are put into tiers by SOURCE_WEIGHT: highest weight = tier 1; origins with EQUAL weight share a tier
# - Per sample: tier 1 covers every compound in the sample it can; tier 2 covers what is left; and so on
# - Within a tier, sources are chosen exactly as before (GREEDY / Beam Search with cost = P + alpha * penalty,
#   where the penalty is computed against the full sample, not only the compounds left for that tier)
# - Beam Search alternatives are found per tier and combined into the best overall covers (lowest total cost)
# - This cell, if used, replaces greedy_cover and beam_search_covers by tiered versions with the same inputs & outputs,
#   so alpha tuning, all validation cells and the per-sample analysis automatically use the tiered cover
# - Set USE_TIERED_COVER = False and re-run the notebook to get the original (non-tiered) results,
#   e.g., as a sensitivity analysis
# - IMPORTANT: this cell must stay directly after the "Beam Search multiple-cover algorithm" cell
# - If you edit & re-run the GREEDY or Beam Search cell, re-run this cell afterwards

USE_TIERED_COVER = False

# Keep references to the original (non-tiered) algorithms (safe to re-run this cell)
if not getattr(greedy_cover, "is_tiered_wrapper", False):
    greedy_cover_plain = greedy_cover
if not getattr(beam_search_covers, "is_tiered_wrapper", False):
    beam_search_covers_plain = beam_search_covers

# Tier order from SOURCE_WEIGHT (highest first; equal weights → same tier)
tier_weights = sorted(set(SOURCE_WEIGHT.values()), reverse=True)
SOURCE_TIERS = [sorted(o for o in SOURCE_WEIGHT if SOURCE_WEIGHT[o] == w) for w in tier_weights]
ORIGIN_TO_TIER = {o: t for t, origins in enumerate(SOURCE_TIERS) for o in origins}
N_TIERS = len(SOURCE_TIERS) + 1 # last tier catches origins missing from SOURCE_WEIGHT (should stay empty)

logging.info(f"Source tiers (first = highest priority):")
for t, origins in enumerate(SOURCE_TIERS, start=1):
    logging.info(f" - Tier {t} (weight {tier_weights[t-1]}): {', '.join(origins)}")

unknown_origins = sorted(set(source_meta_unique["source_origin"].dropna()) - set(ORIGIN_TO_TIER))
if unknown_origins:
    logging.warning(f" - Origins not in SOURCE_WEIGHT (put in a last tier): {unknown_origins}")
minorsep()

def split_into_tiers(source_index):
    # Returns, per tier, the positions (in source_index / A) of the sources belonging to that tier
    tiers = [[] for _ in range(N_TIERS)]
    for i, sid in enumerate(source_index):
        origin = source_metadata_lookup_dict[sid]["origin"]
        tiers[ORIGIN_TO_TIER.get(origin, N_TIERS - 1)].append(i)
    return tiers

def tier_subproblem(A, idx_tier, remaining):
    # Restrict the sources of one tier to the compounds that are still uncovered
    idx_sub, A_sub = [], []
    for i in idx_tier:
        elems = A[i] & remaining
        if elems:
            idx_sub.append(i)
            A_sub.append(elems)
    return idx_sub, A_sub

# Tiered GREEDY: same inputs/outputs as greedy_cover → returns (x, e)
def tiered_greedy_cover(A, U, P, source_index, alpha=0.0):
    U_set = set(U)
    source_cost = calculate_source_costs(source_index, U_set, P, alpha) # full cost, penalty vs. full sample
    remaining = set(U_set)
    selected = []

    for idx_tier in split_into_tiers(source_index):
        if not remaining:
            break

        idx_sub, A_sub = tier_subproblem(A, idx_tier, remaining)

        if not A_sub:
            continue

        U_sub = sorted(set().union(*A_sub))
        # alpha=0.0 because source_cost already contains alpha * penalty
        x_sub, _ = greedy_cover_plain(A_sub, U_sub, source_cost[idx_sub], [source_index[i] for i in idx_sub], alpha=0.0)
        selected.extend(idx_sub[j] for j in np.where(x_sub == 1)[0])
        remaining -= set(U_sub)

    assert not remaining, "Tiered GREEDY left compounds uncovered"

    x_out = np.zeros(len(A))
    x_out[selected] = 1
    e = np.full(len(U), np.nan) # per-compound coverage cost is not computed in tiered mode (not used downstream)

    return x_out, e

# Tiered Beam Search: same inputs/outputs as beam_search_covers → returns [(x_set, coverage, cost), ...]
def tiered_beam_covers(A, U, P, source_index, alpha=0.0, max_covers=50, **beam_kwargs):
    U_set = set(U)
    source_cost = calculate_source_costs(source_index, U_set, P, alpha)
    remaining = set(U_set)
    combos = [(0.0, [])] # (total cost, selected positions) of the best partial covers so far

    for idx_tier in split_into_tiers(source_index):
        if not remaining:
            break

        idx_sub, A_sub = tier_subproblem(A, idx_tier, remaining)

        if not A_sub:
            continue

        U_sub = sorted(set().union(*A_sub))
        cost_sub = source_cost[idx_sub]
        ids_sub = [source_index[i] for i in idx_sub]

        # Alternative covers for this tier: GREEDY + Beam Search (alpha=0.0: cost_sub already contains the penalty)
        tier_covers = {}

        x_g, _ = greedy_cover_plain(A_sub, U_sub, cost_sub, ids_sub, alpha=0.0)
        key = tuple(sorted(int(j) for j in np.where(x_g == 1)[0]))
        tier_covers[key] = float(cost_sub[list(key)].sum())

        for x_set, _, _ in beam_search_covers_plain(A_sub, U_sub, cost_sub, ids_sub, alpha=0.0, max_covers=max_covers, **beam_kwargs):
            key = tuple(sorted(int(j) for j in x_set))
            tier_covers[key] = float(cost_sub[list(key)].sum())

        tier_list = [(c, [idx_sub[j] for j in key]) for key, c in tier_covers.items()]

        # Combine with previous tiers; keep the max_covers cheapest combinations (exact top-k)
        combos = sorted(
            ((c0 + c1, sel0 + sel1) for c0, sel0 in combos for c1, sel1 in tier_list),
            key=lambda t: (t[0], sorted(t[1]))
        )[:max_covers]

        remaining -= set(U_sub)

    assert not remaining, "Tiered Beam Search left compounds uncovered"

    return [(sel, np.full(len(U), np.nan), cost) for cost, sel in combos]

tiered_greedy_cover.is_tiered_wrapper = True
tiered_beam_covers.is_tiered_wrapper = True

# Switch
if USE_TIERED_COVER:
    greedy_cover = tiered_greedy_cover
    beam_search_covers = tiered_beam_covers
    logging.info(f"TIERED cover is ON (greedy_cover & beam_search_covers now use tiers)")
else:
    greedy_cover = greedy_cover_plain
    beam_search_covers = beam_search_covers_plain
    logging.info(f"TIERED cover is OFF (original non-tiered cover)")

minorsep()
logging.info(f"Done defining tiered set cover | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Frequency-weighted compound penalty lookup
majorsep()
logging.info(f"Building compound-frequency lookup | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Number of unique source profiles containing each compound
compound_frequency = (
    sources_masked
    .groupby("inchikey")["source_id"]
    .nunique()
    .to_dict()
)

# Penalty weight:
# common compounds → small penalty
# rare compounds → large penalty
# spreads penalties over wider range instead of collapsing most common compounds penalties toward 0
compound_penalty = {ik: 1.0 / np.sqrt(freq) for ik, freq in compound_frequency.items()}

PENALTY_MIN = min(compound_penalty.values())
PENALTY_MAX = max(compound_penalty.values())
PENALTY_RANGE = max(PENALTY_MAX - PENALTY_MIN, 1e-12)

logging.info(f" - Unique compounds: {len(compound_penalty):,}")
logging.info(f" - Compound frequency range: {min(compound_frequency.values())} – {max(compound_frequency.values()):,}")
logging.info(f" - Mean frequency: {np.mean(list(compound_frequency.values())):.0f}")
logging.info(f" - Median frequency: {np.median(list(compound_frequency.values())):.0f}")
logging.info(f" - Mean penalty: {np.mean(list(compound_penalty.values())):.4f}")
logging.info(f" - Median penalty: {np.median(list(compound_penalty.values())):.4f}")
minorsep()

# Penalty diagnostics
penalty_df = (
    pd.DataFrame({
        "inchikey": list(compound_frequency.keys()),
        "frequency": list(compound_frequency.values()),
        "penalty": [compound_penalty[k] for k in compound_frequency]
    })
)

logging.info(f"Frequency - Lowest:")
display(penalty_df.sort_values("frequency").head(5))
minorsep()
logging.info(f"Frequency - Highest:")
display(penalty_df.sort_values("frequency", ascending=False).head(5))
minorsep()
logging.info(f"Frequency stats:")
logging.info(penalty_df["frequency"].describe())
minorsep()
logging.info(f"Penalty stats:")
logging.info(penalty_df["penalty"].describe())

minorsep()
logging.info(f"Built compound-frequency lookup | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Alpha tuning for frequency-penalized GREEDY set cover
majorsep()
logging.info(f"Starting alpha tuning for GREEDY | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

alphas = np.logspace(3, 5, 10) # 1000 to 100000
n_reps = 50 # keep manageable for tuning

source_ids_all = list(source_cpds.keys())
alpha_scores = []

# Evaluate alpha on synthetic mixtures; uses compound precision as primary metric
def score_alpha(alpha, n_true=50):
    rng = np.random.default_rng(GLOBAL_SEED) # local - good: less noisier comparisons
    scores = []

    for rep in range(n_reps):
        true_sources = rng.choice(source_ids_all, size=n_true, replace=False)
        true_compounds = set().union(*(source_cpds[sid] for sid in true_sources))
        
        U = list(true_compounds)
        U_set = set(U)

        A, source_index, P = build_problem(U_set, household_id=None, uncertainty_sigma=0.0)
        if A is None:
            continue
        
        # inject alpha + frequency penalty
        x, _ = greedy_cover(A, U, P, source_index, alpha=alpha)

        recovered = {source_index[i] for i in np.where(x == 1)[0]}
        recovered_compounds = set().union(*(source_cpds[sid] for sid in recovered)) if recovered else set()
        
        precision = len(recovered_compounds & true_compounds) / len(recovered_compounds) if recovered_compounds else 0
        expansion = len(recovered_compounds) / len(true_compounds)
        score = precision - 0.25 * (expansion - 1) # choose fewer sources
        scores.append(score)

    return np.mean(scores)

for alpha in tqdm(alphas, desc="Alpha tuning for GREEDY"):
    # use fixed mixture size for tuning stability
    s = score_alpha(alpha, n_true=50)
    alpha_scores.append({"alpha": alpha, "score": s})

alpha_df = pd.DataFrame(alpha_scores)

ALF_DIR = Path(DATA_DIR / "alpha_tuning")
ALF_DIR.mkdir(exist_ok=True)
alpha_df.to_csv(ALF_DIR / "alpha_greedy.csv", index=False)
minorsep()
logging.info(f"alpha_greedy saved to: \n{ALF_DIR}")
minorsep()

best_alpha = alpha_df.loc[alpha_df["score"].idxmax(), "alpha"]
logging.info(f"Best alpha GREEDY: {best_alpha.round(2)}")

minorsep()
logging.info(f"Done alpha tuning for GREEDY | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Alpha tuning for Beam Search
majorsep()
logging.info(f"Starting alpha tuning for Beam Search | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

alphas_beam = np.logspace(3, 5, 10) # 1000 to 100000
n_reps = 50

source_ids_all = list(source_cpds.keys())
alpha_scores_beam = []

def score_alpha_beam(alpha, n_true=50):
    rng = np.random.default_rng(GLOBAL_SEED)
    scores = []

    for rep in range(n_reps):
        true_sources = rng.choice(source_ids_all, size=n_true, replace=False)
        true_compounds = set().union(*(source_cpds[sid] for sid in true_sources))

        U = list(true_compounds)
        U_set = set(U)

        A, source_index, P = build_problem(U_set, household_id=None, uncertainty_sigma=0.0)

        if A is None:
            continue

        covers_found = beam_search_covers(
            A,
            U,
            P,
            source_index,
            alpha=alpha,
            # beam_width=50,
            beam_width=15,
            # max_covers=50,
            max_covers=20,
            diversity_lambda=0.15
        )

        if not covers_found:
            continue

        best_cover_indices, _, _ = covers_found[0]
        recovered = {source_index[i] for i in best_cover_indices}
        recovered_compounds = set().union(*(source_cpds[sid] for sid in recovered)) if recovered else set()
        precision = len(recovered_compounds & true_compounds) / len(recovered_compounds) if recovered_compounds else 0
        expansion = len(recovered_compounds) / len(true_compounds)
        score = precision - 0.25 * (expansion - 1)

        scores.append(score)

    return np.mean(scores) if scores else np.nan

for alpha in tqdm(alphas_beam, desc="Alpha tuning for Beam Search"):
    s = score_alpha_beam(alpha)
    alpha_scores_beam.append({"alpha": alpha, "score": s})

alpha_df_beam = pd.DataFrame(alpha_scores_beam)

ALF_DIR = Path(DATA_DIR / "alpha_tuning")
ALF_DIR.mkdir(exist_ok=True)
alpha_df_beam.to_csv(ALF_DIR / "alpha_beam.csv", index=False)

best_alpha_beam = alpha_df_beam.loc[alpha_df_beam.score.idxmax(), "alpha"]
minorsep()
logging.info(f"Best alpha Beam Search: {best_alpha_beam.round(2)}")

minorsep()
logging.info(f"Done alpha tuning for Beam Search | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Load best alphas for GREEDY & Beam Search
majorsep()
logging.info(f"Loading best alphas for GREEDY & Beam Search | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

alpha_df = pd.read_csv(ALF_DIR / "alpha_greedy.csv")
alpha_df = alpha_df.sort_values("alpha")
best_alpha = alpha_df.loc[alpha_df["score"].idxmax(), "alpha"]
logging.info(f" - Best alpha GREEDY:      {best_alpha.round(2)}")

alpha_df_beam = pd.read_csv(ALF_DIR / "alpha_beam.csv")
alpha_df_beam = alpha_df_beam.sort_values("alpha")
best_alpha_beam = alpha_df_beam.loc[alpha_df_beam["score"].idxmax(), "alpha"]
logging.info(f" - Best alpha Beam Search: {best_alpha_beam.round(2)}")

minorsep()
logging.info(f"Done loading best alphas for GREEDY & Beam Search | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Plot best alphas comparison
majorsep()
logging.info(f"Plotting combined alpha tuning curves | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

logging.info(f"Best alpha GREEDY:      {best_alpha.round(2)}")
logging.info(f"Best alpha Beam Search: {best_alpha_beam.round(2)}")
minorsep()

plt.figure(figsize=(5, 7))
plt.plot(alpha_df["alpha"], alpha_df["score"], marker="o", label="GREEDY")
plt.plot(alpha_df_beam["alpha"], alpha_df_beam["score"], marker="s", label="Beam Search")
plt.xscale("log")
plt.xlabel("alpha (penalty strength)")
plt.ylabel("score")
plt.title("Alpha tuning: GREEDY vs. Beam Search")
plt.grid(True)
plt.legend()

plt.savefig(ALF_DIR / "alphas_greedy_vs_beam.png", dpi=600, bbox_inches="tight")
plt.show()

minorsep()
logging.info(f"Plotted combined alpha tuning curves | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Plot best alphas comparison (non-log)
majorsep()
logging.info(f"Plotting combined alpha tuning curves (non-log) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

from matplotlib.ticker import MultipleLocator, ScalarFormatter

plt.figure(figsize=(10, 5))

plt.plot(alpha_df["alpha"], alpha_df["score"], marker="o", label="GREEDY")
plt.plot(alpha_df_beam["alpha"], alpha_df_beam["score"], marker="s", label="Beam Search")

plt.xlabel("alpha (penalty strength)")
plt.ylabel("score")
plt.title("Alpha tuning: GREEDY vs. Beam Search")

# Restrict x-axis and set ticks
plt.xlim(500, 10000)
plt.gca().xaxis.set_major_locator(MultipleLocator(500))
plt.gca().xaxis.set_major_formatter(ScalarFormatter())
plt.ticklabel_format(style="plain", axis="x")

plt.grid(True)
plt.legend()

plt.savefig(ALF_DIR / "alphas_greedy_vs_beam_non-log.png", dpi=600, bbox_inches="tight")
plt.show()

minorsep()
logging.info(f"Plotted combined alpha tuning curves (non-log) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# OPTIONAL: Specify best alphas manually based on the plots
majorsep()

# best_alpha = 4500
# best_alpha_beam = 4500

logging.info(f"Best alphas to be used:")
logging.info(f" - best_alpha GREEDY:      {best_alpha.round(2)}")
logging.info(f" - best_alpha Beam Search: {best_alpha_beam.round(2)}")
majorsep()


In [ ]:
# Separate random seed for validation
majorsep()

# Alpha tuning uses GLOBAL_SEED to draw its synthetic mixtures. If validation used the same seed,
# some validation mixtures would be identical to the tuning mixtures (tuning and testing on the same data)
# All validation cells below now use VALIDATION_SEED instead
VALIDATION_SEED = GLOBAL_SEED + 1000

logging.info(f"Validation seed: {VALIDATION_SEED} \nTuning seed:     {GLOBAL_SEED}")
majorsep()


In [ ]:
# Calibrate prior strength (balance between priors and the frequency penalty)
majorsep()
logging.info(f"Calibrating prior strength | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Why: a source's cost = PRIOR_STRENGTH * (-log prior) + coverage term + alpha * penalty
# - With the old fixed PRIOR_STRENGTH = 2.0, the priors (source weights, categories, geo metadata, household relation)
#   spanned ~8 cost units and the penalty ~150, so the priors hardly mattered
# - Here, PRIOR_STRENGTH is set so that, on the real samples, the spread of the prior term (5th-95th percentile
#   across candidate sources) equals PRIOR_TO_PENALTY_RATIO × the spread of the penalty term
# - PRIOR_TO_PENALTY_RATIO = 1.0 → priors and penalty have equal weight (recommended default)
#                            0.5 → penalty (chemical fit) twice as strong
#                            2.0 → priors twice as strong
# - Report the value used; running 0.5 & 2.0 as a sensitivity analysis is a good idea
PRIOR_TO_PENALTY_RATIO = 1.0
# N samples used for calibration
CALIBRATION_MAX_SAMPLES = len([c for c in sample_universes_df.columns if c != "inchikey"]) # all available samples

masked_inchikeys = set(sources_masked["inchikey"])
calib_samples = sorted(c for c in sample_universes_df.columns if c != "inchikey")[:CALIBRATION_MAX_SAMPLES]
calib_rows = []

for s in tqdm(calib_samples, desc="Prior strength calibration"):
    U_set = set(sample_universes_df[s].dropna()) & masked_inchikeys

    if not U_set:
        continue

    household_id = household_key(s)
    cal_index = [sid for sid, cpds in source_cpds.items() if cpds & U_set]

    # -log(prior) per candidate source (household-aware: geo metadata & household relation included)
    neg_log_prior = -np.log(np.clip([source_prior(sid, household_id=household_id) for sid in cal_index], 1e-6, None))

    # alpha * penalty per candidate source (P = 0 → calculate_source_costs returns only the penalty term)
    penalty_term = calculate_source_costs(cal_index, U_set, np.zeros(len(cal_index)), best_alpha)

    calib_rows.append({
        "sample": s,
        "neg_log_prior_range": np.percentile(neg_log_prior, 95) - np.percentile(neg_log_prior, 5),
        "penalty_range": np.percentile(penalty_term, 95) - np.percentile(penalty_term, 5)
    })

calib_df = pd.DataFrame(calib_rows)
prior_range = calib_df["neg_log_prior_range"].median()
penalty_range = calib_df["penalty_range"].median()

if prior_range > 0 and penalty_range > 0:
    PRIOR_STRENGTH = float(PRIOR_TO_PENALTY_RATIO * penalty_range / prior_range)
else:
    PRIOR_STRENGTH = 2.0
    logging.warning(f" - Could not calibrate (zero spread); PRIOR_STRENGTH kept at 2.0")

minorsep()
logging.info(f" - Samples used: {len(calib_df):,}")
logging.info(f" - Median spread of -log(prior): {prior_range:.3f}")
logging.info(f" - Median spread of alpha * penalty (alpha = {best_alpha}): {penalty_range:.1f}")
logging.info(f" - PRIOR_TO_PENALTY_RATIO: {PRIOR_TO_PENALTY_RATIO}")
logging.info(f" - PRIOR_STRENGTH set to: {PRIOR_STRENGTH:.2f} (was 2.0)")
logging.info(f" - All following cells (validation & per-sample analysis) use this PRIOR_STRENGTH")

ALF_DIR.mkdir(exist_ok=True)
calib_df.to_csv(ALF_DIR / "prior_strength_calibration.csv", index=False)

minorsep()
logging.info(f"Done calibrating prior strength | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Synthetic perturbation validation
majorsep()
logging.info(f"Running synthetic robustness validation (dropout & contamination) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

rng = np.random.default_rng(VALIDATION_SEED)

VALIDATE_REAL = True
perturb_results = []

# Perturbation strengths
dropout_rates = [0.05, 0.1, 0.2]
contam_rates = [0.0, 0.05, 0.1]

source_ids_all = list(source_cpds.keys())

def perturb_sources(true_sources, dropout=0.1, contam=0.05):
    perturbed = set()
    all_compounds = sorted({c for s in source_cpds.values() for c in s}) # sorted → reproducible between kernel restarts

    for sid in true_sources:
        comps = sorted(source_cpds[sid]) # sorted → reproducible between kernel restarts

        # dropout
        kept = {c for c in comps if rng.random() > dropout}

        # contamination
        n_contam = int(contam * len(comps))
        contam_added = set(rng.choice(all_compounds, size=n_contam, replace=False))
        perturbed |= kept | contam_added

    return perturbed

if VALIDATE_REAL:
    outer_iter = [(d, c) for d in dropout_rates for c in contam_rates]

    for dropout, contam in tqdm(outer_iter, desc="Dropout × contamination"):
        for rep in tqdm(range(100), desc=f"rep (d={dropout}, c={contam})", leave=False):
            true_sources = rng.choice(source_ids_all, size=50, replace=False)

            true_compounds = set()
            for sid in true_sources:
                true_compounds |= source_cpds[sid]

            U = sorted(true_compounds)
            U_set = set(U)

            perturbed_U = perturb_sources(true_sources, dropout, contam)

            A, source_index, P = build_problem(perturbed_U, household_id=None)
            if A is None:
                continue
                
            x, _ = greedy_cover(A, list(perturbed_U), P, source_index, alpha=best_alpha)

            recovered = {source_index[i] for i in np.where(x == 1)[0]}
            false_sources = recovered - set(true_sources)
            true_sources_set = set(true_sources)
            
            source_precision = (
                len(recovered & true_sources_set) / len(recovered)
                if recovered else 0
            )
            
            false_discovery_rate = (
                len(false_sources) / len(recovered)
                if recovered else 0
            )

            recovered_compounds = set()
            for sid in recovered:
                recovered_compounds |= source_cpds[sid]

            precision = (
                len(recovered_compounds & true_compounds)
                / len(recovered_compounds)
                if recovered_compounds else 0
            )

            recall = (
                len(recovered_compounds & true_compounds)
                / len(true_compounds)
                if true_compounds else 0
            )

            perturb_results.append({
                "dropout": dropout,
                "contamination": contam,
                "precision": precision,
                "recall": recall,
                "source_precision": source_precision,
                "false_discovery_rate": false_discovery_rate
            })

perturb_df = pd.DataFrame(perturb_results)

summary_perturb = perturb_df.groupby(["dropout", "contamination"]).mean()
minorsep()
display(summary_perturb.round(3))

minorsep()
logging.info(f"Completed synthetic perturbation validation (dropout & contamination) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Synthetic recovery validation with overlap sensitivity
majorsep()
logging.info(f"Running overlap-sensitivity synthetic validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

VALIDATE_SYNTHETIC = True

validation_results = []

rng = np.random.default_rng(VALIDATION_SEED)

source_ids_all = list(source_cpds.keys())
mixture_sizes = [10, 100, 200, 300]
replicates = 100
overlap_thresholds = [0.10, 0.50, "random"]

def choose_sources_overlap_control(n_sources, max_jaccard=None):
    candidates = list(source_cpds.keys())
    rng.shuffle(candidates)
    selected = []

    for sid in candidates:
        if len(selected) == 0:
            selected.append(sid)
            continue

        if max_jaccard is None:
            # random mode: no overlap constraint at all
            selected.append(sid)
        else:
            current = source_cpds[sid]
        
            ok = True
        
            for chosen in selected:
                other = source_cpds[chosen]
                inter = len(current & other)
                union = len(current | other)
                j = inter / union if union else 0.0
        
                if j > max_jaccard:
                    ok = False
                    break
        
            if ok:
                selected.append(sid)

        if len(selected) == n_sources:
            break

    return selected


if VALIDATE_SYNTHETIC:
    outer = [(o, n) for o in overlap_thresholds for n in mixture_sizes]

    for overlap, n_true in tqdm(outer, desc="Overlap × mixture size"):
        for rep in tqdm(range(replicates), desc=f"rep (o={overlap}, n={n_true})", leave=False):
            max_jaccard = None if overlap == "random" else float(overlap)
            true_sources = choose_sources_overlap_control(n_true, max_jaccard)

            # record when the overlap constraint could not be met and a random mixture was used instead
            fallback_used = len(true_sources) < n_true
            if fallback_used:
                true_sources = rng.choice(source_ids_all, size=n_true, replace=False)
            
            true_compounds = set()
            
            for sid in true_sources:
                true_compounds |= source_cpds[sid]

            U = list(true_compounds)

            A, source_index, P = build_problem(true_compounds, household_id=None)
            
            if A is None:
                continue
            
            x, _ = greedy_cover(A, U, P, source_index, alpha=best_alpha)

            recovered = {source_index[i] for i in np.where(x == 1)[0]}
            true_sources = set(true_sources)

            tp = len(recovered & true_sources)
            fp = len(recovered - true_sources)
            
            source_precision = (tp / len(recovered) if recovered else 0)
            source_recall = (tp / len(true_sources) if true_sources else 0)
            
            source_f1 = (
                2 * source_precision * source_recall /
                (source_precision + source_recall)
                if (source_precision + source_recall) > 0 else 0
            )
            
            false_discovery_rate = (
                fp / len(recovered)
                if recovered else 0
            )

            recovered_compounds = set()
            
            for sid in recovered:
                recovered_compounds |= source_cpds[sid]

            precision = (
                len(recovered_compounds & true_compounds)
                / len(recovered_compounds)
                if recovered_compounds else 0
            )

            validation_results.append({
                "overlap": overlap,
                "fallback_random": fallback_used, # mean of this column = fraction of random fallback mixtures
                "n_true_sources": n_true,
                "precision": precision,
                "source_precision": source_precision,
                "source_recall": source_recall,
                "source_f1": source_f1,
                "false_discovery_rate": false_discovery_rate,
                "n_selected_sources": len(recovered),
                "tp_sources": tp,
                "fp_sources": fp
            })

overlap_validation_df = pd.DataFrame(validation_results)
overlap_summary = overlap_validation_df.groupby(["overlap", "n_true_sources"]).mean()

minorsep()
logging.info(f"Random fallback mixtures (overlap constraint not met): {int(overlap_validation_df['fallback_random'].sum()):,} of {len(overlap_validation_df):,} replicates")
logging.info(f"Rows with fallback_random > 0 in the table below are (partly) random mixtures, not overlap-controlled ones")

minorsep()
display(overlap_summary.round(3))
minorsep()

logging.info(f"Completed overlap sensitivity analysis | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Synthetic recovery validation (chemistry-based)
majorsep()
logging.info(f"Running synthetic recovery validation (chemistry-based) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

VALIDATE_SYNTHETIC = True

validation_results = []

if VALIDATE_SYNTHETIC:
    rng = np.random.default_rng(VALIDATION_SEED)
    source_ids_all = list(source_cpds.keys())
    mixture_sizes = [10, 100, 200, 300]
    replicates = 100

    def choose_distinct_sources(source_cpds, n_sources, rng, max_jaccard=0.25):
        candidates = list(source_cpds.keys())
        rng.shuffle(candidates)

        selected = []

        for sid in candidates:
            current = source_cpds[sid]
            keep = True

            for chosen in selected:
                other = source_cpds[chosen]
                inter = len(current & other)
                union = len(current | other)
                j = inter / union if union else 0

                if j > max_jaccard:
                    keep = False
                    break

            if keep:
                selected.append(sid)

            if len(selected) == n_sources:
                break

        return selected

    for n_true in mixture_sizes:
        for rep in tqdm(range(replicates), desc=f"Synthetic mixtures ({n_true} sources & {replicates} replicates)"):
            true_sources = choose_distinct_sources(source_cpds, n_true, rng, max_jaccard=0.25)

            # record when the Jaccard constraint could not be met and a random mixture was used instead
            fallback_used = len(true_sources) < n_true
            if fallback_used:
                true_sources = rng.choice(source_ids_all, size=n_true, replace=False)

            true_sources_set = set(true_sources)

            true_compounds = set()
            for sid in true_sources:
                true_compounds.update(source_cpds[sid])

            U = sorted(true_compounds)

            if len(U) == 0:
                continue

            U_set = set(U)

            A, source_index, P = build_problem(U_set, household_id=None)
            if A is None:
                continue
            
            x, e = greedy_cover(A, U, P, source_index, alpha=best_alpha)

            recovered_sources = {source_index[i] for i in np.where(x == 1)[0]}
            recovered_sources_set = set(recovered_sources)
            recovered_compounds = set()
            
            for sid in recovered_sources:
                recovered_compounds.update(source_cpds[sid])

            # Compound-level metrics
            intersection = len(true_compounds & recovered_compounds)
            union = len(true_compounds | recovered_compounds)
            compound_precision = (intersection / len(recovered_compounds) if recovered_compounds else 0)
            compound_recall = (intersection / len(true_compounds) if true_compounds else 0)

            compound_f1 = (
                2 * compound_precision * compound_recall /
                (compound_precision + compound_recall)
                if (compound_precision + compound_recall) > 0 else 0
            )

            jaccard = intersection / union if union else 0

            expansion_factor = (
                len(recovered_compounds) / len(true_compounds)
                if true_compounds else np.nan
            )

            # Source-level metrics
            source_precision = (
                len(recovered_sources_set & true_sources_set)
                / len(recovered_sources_set)
                if recovered_sources_set else 0
            )

            source_recall = (
                len(recovered_sources_set & true_sources_set)
                / len(true_sources_set)
                if true_sources_set else 0
            )

            false_discovery_rate = (
                len(recovered_sources_set - true_sources_set)
                / len(recovered_sources_set)
                if recovered_sources_set else 0
            )

            validation_results.append({
                "n_true_sources": n_true,
                "fallback_random": fallback_used,
                # compound-level
                "compound_precision": compound_precision,
                "compound_recall": compound_recall,
                "compound_f1": compound_f1,
                "jaccard": jaccard,
                "compound_expansion_factor": expansion_factor,
                # source-level
                "source_precision": source_precision,
                "source_recall": source_recall,
                "false_discovery_rate": false_discovery_rate,
                # diagnostics
                "n_true_compounds": len(true_compounds),
                "n_recovered_compounds": len(recovered_compounds),
                "n_recovered_sources": len(recovered_sources_set),
            })

synth_validation_df = pd.DataFrame(validation_results)

synth_summary = (
    synth_validation_df
    .groupby("n_true_sources")
    .agg(
        compound_precision=("compound_precision", "mean"),
        compound_recall=("compound_recall", "mean"),
        compound_f1=("compound_f1", "mean"),
        jaccard=("jaccard", "mean"),
        compound_expansion_factor=("compound_expansion_factor", "mean"),
        source_precision=("source_precision", "mean"),
        source_recall=("source_recall", "mean"),
        false_discovery_rate=("false_discovery_rate", "mean"),
        fallback_random_rate=("fallback_random", "mean"), # fraction of random fallback mixtures
        mean_true_compounds=("n_true_compounds", "mean"),
        mean_recovered_compounds=("n_recovered_compounds", "mean"),
        mean_recovered_sources=("n_recovered_sources", "mean"),
    )
)

minorsep()
logging.info(f"Synthetic validation summary:")
display(synth_summary.round(3))

minorsep()
logging.info(f"Done running synthetic recovery validation (chemistry-based) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Synthetic recovery validation (chemistry-based) summary statistics
synth_summary_2 = (
    synth_validation_df
    .groupby("n_true_sources")
    .agg(
        mean_precision=("compound_precision", "mean"),
        sd_precision=("compound_precision", "std"),
        n=("compound_precision", "count"),
        mean_recovered_sources=("n_recovered_sources", "mean")
    )
)

# Standard error
synth_summary_2["se_precision"] = (synth_summary_2["sd_precision"] / np.sqrt(synth_summary_2["n"]))

# 95% confidence interval
synth_summary_2["precision_ci_low"] = (synth_summary_2["mean_precision"] - 1.96 * synth_summary_2["se_precision"])
synth_summary_2["precision_ci_high"] = (synth_summary_2["mean_precision"] + 1.96 * synth_summary_2["se_precision"])

# Keep values within [0,1]
synth_summary_2["precision_ci_low"] = synth_summary_2["precision_ci_low"].clip(lower=0)
synth_summary_2["precision_ci_high"] = synth_summary_2["precision_ci_high"].clip(upper=1)

majorsep()
logging.info("Synthetic validation summary:")
minorsep()
display(synth_summary_2.round(3))
majorsep()


In [ ]:
# Synthetic recovery validation plot
majorsep()
logging.info(f"Plotting synthetic recovery validation results | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Summary statistics for plotting
plot_df = synth_summary_2.reset_index()

# Error bars
lower_err = (plot_df["mean_precision"] - plot_df["precision_ci_low"])
upper_err = (plot_df["precision_ci_high"] - plot_df["mean_precision"])

fig, axes = plt.subplots(1, 2, figsize=(7, 3), constrained_layout=True)

# Left panel
axes[0].errorbar(
    plot_df["n_true_sources"],
    plot_df["mean_precision"],
    yerr=[lower_err, upper_err],
    fmt="o-",
    linewidth=2,
    capsize=5,
    markersize=7
)

axes[0].set_xlabel("True sources in synthetic mixture")
axes[0].set_ylabel("Compound precision")
axes[0].set_ylim(0, 1.02)
axes[0].set_xticks(plot_df["n_true_sources"])
axes[0].grid(alpha=0.3)
axes[0].set_title("A", loc="left", fontweight="bold")

# Right panel
axes[1].plot(
    plot_df["n_true_sources"],
    plot_df["mean_recovered_sources"],
    "o-",
    linewidth=2,
    markersize=7
)

axes[1].set_xlabel("True sources in synthetic mixture")
axes[1].set_ylabel("Mean recovered sources")
axes[1].set_xticks(plot_df["n_true_sources"])
axes[1].grid(alpha=0.3)
axes[1].set_title("B", loc="left", fontweight="bold")

plt.savefig(VAL_DIR / "synthetic_validation_precision.png", dpi=600, bbox_inches="tight")
plt.show()

minorsep()
logging.info(f"Plotted synthetic recovery validation results | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Synthetic recovery validation results plot
majorsep()
logging.info(f"Plotting synthetic recovery validation results | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

mixture_sizes = sorted(synth_validation_df["n_true_sources"].unique())

data = [
    synth_validation_df.loc[
        synth_validation_df["n_true_sources"] == n,
        "compound_precision"
    ].values
    for n in mixture_sizes
]

plt.figure(figsize=(5, 3))
plt.boxplot(data, tick_labels=mixture_sizes, showfliers=False)
plt.xlabel("True sources in synthetic mixture")
plt.ylabel("Compound precision")
plt.ylim(0, 1.02)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(VAL_DIR / "synthetic_validation_boxplot.png", dpi=600, bbox_inches="tight")
plt.show()

minorsep()
logging.info(f"Plotted synthetic recovery validation results | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Cost-consistent validation
majorsep()
logging.info(f"Running cost-consistent synthetic validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

cost_validation = []

rng = np.random.default_rng(VALIDATION_SEED)

mixture_sizes = [10, 100, 200, 300]
replicates = 100

source_ids_all = list(source_cpds.keys())

# True sources are sampled proportionally to their household-independent prior,
# i.e. sources the model considers more plausible are drawn more often
prior_weights = np.array([source_prior_lookup[s] for s in source_ids_all], dtype=float)
weights = prior_weights / prior_weights.sum()

for n_true in mixture_sizes:
    for rep in tqdm(range(replicates), desc=f"Cost-consistent (mixture size {n_true})"):
        true_sources = rng.choice(
            source_ids_all,
            size=n_true,
            replace=False,
            p=weights
        )

        true_sources = list(true_sources)
        true_sources_set = set(true_sources)
        true_compounds = set()

        for sid in true_sources:
            true_compounds |= source_cpds[sid]

        U = sorted(true_compounds)

        A, source_index, P = build_problem(true_compounds, household_id=None)
        if A is None:
            continue

        # compare costs on the same objective the optimiser minimises (prior/coverage cost + alpha * penalty)
        full_cost = calculate_source_costs(source_index, set(U), P, best_alpha)
        cost_lookup = dict(zip(source_index, full_cost))

        x, _ = greedy_cover(A, U, P, source_index, alpha=best_alpha)

        recovered_sources = {source_index[i] for i in np.where(x == 1)[0]}
        recovered_compounds = set()

        for sid in recovered_sources:
            recovered_compounds |= source_cpds[sid]

        intersection = len(recovered_compounds & true_compounds)
        compound_precision = (intersection / len(recovered_compounds) if recovered_compounds else 0)
        compound_recall = (intersection / len(true_compounds) if true_compounds else 0)

        source_precision = (len(recovered_sources &
                true_sources_set
            ) / len(recovered_sources)
            if recovered_sources else 0)

        source_recall = (len(recovered_sources &
                true_sources_set
            ) / len(true_sources_set))

        recovered_cost = sum(cost_lookup[s] for s in recovered_sources)
        true_cost = np.sum([cost_lookup[s] for s in true_sources if s in cost_lookup])

        cost_validation.append({
            "n_true_sources": n_true,
            "compound_precision": compound_precision,
            "compound_recall": compound_recall,
            "source_precision": source_precision,
            "source_recall": source_recall,
            "cost_ratio": (recovered_cost / true_cost if true_cost > 0 else np.nan)})

cost_df = pd.DataFrame(cost_validation)
display(cost_df.groupby("n_true_sources").mean().round(3))

minorsep()
logging.info(f"Completed cost-consistent validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Noise robustness validation
majorsep()
logging.info(f"Running noise robustness validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

noise_results = []
missing_rates = [0.1, 0.3, 0.5]
mixture_sizes = [10, 100, 200, 300]
replicates = 100

for n_true_sources in mixture_sizes:
    for missing_rate in missing_rates:
        for rep in range(replicates):
            if rep % 50 == 0:
                tqdm.write(f"{n_true_sources} sources | missing {missing_rate * 100}% | replicates {rep}")
            
            # Generate synthetic mixture
            true_sources = rng.choice(source_ids_all, size=n_true_sources, replace=False)
            true_union = set()

            for sid in true_sources:
                true_union.update(source_cpds[sid])

            if len(true_union) < 10:
                continue

            true_union = set(true_union)

            # Remove compounds (simulate missing detections)
            n_remove = int(len(true_union) * missing_rate)

            if n_remove > 0:
                removed_idx = rng.choice(
                    len(true_union),
                    size=n_remove,
                    replace=False
                )

                removed_idx_set = set(removed_idx.tolist())
                observed = {
                    c for i, c in enumerate(sorted(true_union))
                    if i not in removed_idx_set
                } # sorted → reproducible between kernel restarts
            else:
                observed = true_union.copy()

            U_set = set(observed)

            A, source_index, P = build_problem(U_set, household_id=None)
            if A is None:
                continue
            
            U = sorted(observed)
            x, e = greedy_cover(A, U, P, source_index, alpha=best_alpha)

            recovered_sources = {
                source_index[i]
                for i in np.where(x == 1)[0]
            }

            # Compare compound space
            recovered_union = set()

            for sid in recovered_sources:
                recovered_union.update(source_cpds[sid])

            true_set = set(true_union) # ensure consistent casting
            intersection = len(set(true_union) & recovered_union)
            union = len(set(true_union) | recovered_union)

            jaccard = (len(set(true_union) & recovered_union) / union if union > 0 else 0)

            # Source-level metrics
            true_sources_set = set(true_sources)
            tp = len(recovered_sources & true_sources_set)
            fp = len(recovered_sources - true_sources_set)

            precision = (tp / len(recovered_sources) if len(recovered_sources) > 0 else 0)
            recall = tp / len(true_sources) if len(true_sources) > 0 else 0

            f1 = (
                2 * precision * recall / (precision + recall)
                if (precision + recall) > 0
                else 0
            )

            noise_results.append({
                "n_true_sources": n_true_sources,
                "missing_rate": missing_rate,
                "jaccard": jaccard,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "false_discovery_rate": (
                    fp / len(recovered_sources)
                    if recovered_sources else 0
                ),
                "n_recovered_sources": len(recovered_sources)
            })

noise_df = pd.DataFrame(noise_results)

minorsep()
logging.info(f"Noise robustness validation summary:")

noise_summary = (
    noise_df
    .groupby(["n_true_sources", "missing_rate"])
    .agg(
        mean_jaccard=("jaccard", "mean"),
        median_jaccard=("jaccard", "median"),
        std_jaccard=("jaccard", "std"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1=("f1", "mean"),
        mean_false_discovery_rate=("false_discovery_rate", "mean"),
        mean_recovered_sources=("n_recovered_sources", "mean")
    )
)

display(noise_summary)

minorsep()
logging.info(f"Done running noise robustness validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# GREEDY stability validation
majorsep()
logging.info(f"Running GREEDY stability validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

greedy_stability_results = []
rng = np.random.default_rng(VALIDATION_SEED)

drop_fraction = 0.05
mixture_sizes = [10, 100, 200, 300]
replicates = 100

for n_true in mixture_sizes:
    for rep in tqdm(range(replicates), desc=f"GREEDY stability (mixture size {n_true} & drop {drop_fraction * 100}%)"):
        true_sources = rng.choice(source_ids_all, size=n_true, replace=False)
        compounds = set()

        for sid in true_sources:
            compounds |= source_cpds[sid]

        U = sorted(compounds)

        A, source_index, P = build_problem(compounds, household_id=None)
        if A is None:
            continue
        
        x,_ = greedy_cover(A, U, P, source_index, alpha=best_alpha)

        baseline = {source_index[i] for i in np.where(x == 1)[0]}
        baseline_compounds = set()

        for sid in baseline:
            baseline_compounds |= source_cpds[sid]

        perturbed = set(compounds)

        remove = rng.choice(
            sorted(perturbed), # sorted → reproducible between kernel restarts
            size=int(drop_fraction * len(perturbed)),
            replace=False
        )

        perturbed -= set(remove)
        U2 = sorted(perturbed)
        A2 = []
        source_index2 = []

        for sid,cpds in source_cpds.items():
            elems=cpds & perturbed

            if elems:
                A2.append(elems)
                source_index2.append(sid)

        if len(A2) == 0:
            continue

        P2 = make_costfun(A2, source_index2, source_meta_lookup)
        x2, _ = greedy_cover(A2, U2, P2, source_index2, alpha=best_alpha)
        perturbed_solution = {source_index2[i] for i in np.where(x2==1)[0]}

        # Reconstructed compound space from perturbed solution
        perturbed_compounds = set()
        
        for sid in perturbed_solution:
            perturbed_compounds |= source_cpds[sid]

        compound_union = baseline_compounds | perturbed_compounds

        compound_solution_jaccard = (
            len(baseline_compounds & perturbed_compounds)
            / len(compound_union)
            if len(compound_union) > 0
            else 1.0
        )

        union = baseline | perturbed_solution

        greedy_stability = (
            len(baseline & perturbed_solution)
            / len(union)
            if len(union) > 0
            else 1.0
        )

        greedy_stability_results.append({
            "n_true_sources": n_true,
            "solution_jaccard": greedy_stability,
            "baseline_size": len(baseline),
            "perturbed_size": len(perturbed_solution),
            "compound_solution_jaccard": compound_solution_jaccard
        })

greedy_stability_df = pd.DataFrame(greedy_stability_results)
minorsep()
display(greedy_stability_df.groupby("n_true_sources").mean().round(3))

minorsep()
logging.info(f"Completed GREEDY stability validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Beam Search posterior diagnostics
majorsep()
logging.info(f"Defining beam search posterior diagnostics | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

def posterior_diagnostics(covers):
    if len(covers) == 0:
        return {
            "entropy": np.nan,
            "ess": np.nan,
            "top1": np.nan,
            "top5": np.nan
        }

    costs = np.array([c[2] for c in covers], dtype=float)
    costs -= costs.min()
    posterior = np.exp(-costs)
    posterior /= posterior.sum()
    entropy = -np.sum(posterior * np.log(posterior + 1e-300))
    ess = 1.0 / np.sum(posterior ** 2)
    top1 = posterior.max()
    top5 = np.sort(posterior)[::-1][:5].sum()

    return {
        "entropy": entropy,
        "ess": ess,
        "top1": top1,
        "top5": top5
    }

logging.info(f"Done defining beam search posterior diagnostics | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Beam Search validation
majorsep()
logging.info(f"Running Beam Search validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

beam_widths = [5, 10, 30, 50]
beam_validation = []

rng = np.random.default_rng(VALIDATION_SEED)
source_ids_all = list(source_cpds.keys())

for beam_width in tqdm(beam_widths, desc="Beam widths"):
    runtimes = []
    cover_counts = []
    cover_costs = []
    entropies = []
    ess_values = []
    top1_values = []
    top5_values = []

    for rep in range(20):
        true_sources = rng.choice(source_ids_all, size=50, replace=False)
        U = set().union(*(source_cpds[sid] for sid in true_sources))
        A, source_index, P = build_problem(U, household_id=None, uncertainty_sigma=0.0)

        if A is None:
            continue

        t0 = time.perf_counter()

        covers = beam_search_covers(
            A,
            list(U),
            P,
            source_index,
            alpha=best_alpha_beam,
            beam_width=beam_width,
            max_covers=20
        )

        runtime = time.perf_counter() - t0
        runtimes.append(runtime)
        cover_counts.append(len(covers))

        if covers:
            cover_costs.append(np.mean([c[2] for c in covers]))

            diag = posterior_diagnostics(covers)
            entropies.append(diag["entropy"])
            ess_values.append(diag["ess"])
            top1_values.append(diag["top1"])
            top5_values.append(diag["top5"])
        
    beam_validation.append({
        "beam_width": beam_width,
        "runtime_sec": np.mean(runtimes),
        "covers_found": np.mean(cover_counts),
        "mean_cover_cost": np.mean(cover_costs),
        "posterior_entropy": np.mean(entropies),
        "effective_sample_size": np.mean(ess_values),
        "top1_posterior": np.mean(top1_values),
        "top5_posterior_mass": np.mean(top5_values)
    })

beam_validation_df = pd.DataFrame(beam_validation)
minorsep()
display(beam_validation_df)

minorsep()
logging.info(f"Done Beam Search validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Database redundancy (leave-one-source-out (LOSO))
majorsep()
logging.info(f"Running database redundancy (LOSO) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

loso_results = []

for sid_true in tqdm(source_ids_all, desc="LOSO validation"):
    U = source_cpds[sid_true]

    if len(U) < 5:
        continue

    database_minus_one = {sid: cpds for sid, cpds in source_cpds.items() if sid != sid_true}
    best_jaccard = 0
    best_match = None
    best_containment = 0 # moved here (was reset inside the loop below, so it was almost always 0)

    for sid_test, cpds in database_minus_one.items():
        j = (len(U & cpds) / len(U | cpds))
        
        containment = len(U & cpds) / len(U)
        # best_containment = 0

        if j > best_jaccard:
            best_jaccard = j
            best_containment = containment
            best_match = sid_test

    best_overlap = len(U & database_minus_one[best_match]) if best_match is not None else 0
    
    loso_results.append({
        "source_id": sid_true,
        "best_match": best_match,
        "best_jaccard": best_jaccard,
        "containment": best_containment,
        "n_compounds": len(U),
        "shared_compounds": best_overlap
    })

loso_df = pd.DataFrame(loso_results)
minorsep()
display(loso_df)

minorsep()
logging.info(f"Database redundancy (LOSO) summary:\n")
logging.info(f"Best Jaccard:")
logging.info(loso_df["best_jaccard"].describe())
minorsep()
logging.info(f"Worst Jaccard:")
display(loso_df.nsmallest(10, "best_jaccard"))
minorsep()
logging.info(f"Containment:")
logging.info(loso_df["containment"].describe())
minorsep()
logging.info(f"Worst containment:")
display(loso_df.sort_values("containment").head(10))

minorsep()
logging.info(f"Done running database redundancy (LOSO) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Plot database redundancy LOSO Jaccard values
majorsep()
logging.info(f"Plotting LOSO Jaccard values | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

plt.figure(figsize=(5, 3))
loso_df["best_jaccard"].hist(bins=50)

plt.xlabel("LOSO Jaccard similarity")
plt.ylabel("Frequency")

plt.savefig(VAL_DIR / "loso_jaccard_plot.png", dpi=600, bbox_inches="tight")
plt.show()

minorsep()
logging.info(f"Plotted LOSO Jaccard values | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# LOSO reconstruction validation (algorithm robustness)
majorsep()
logging.info(f"Running LOSO reconstruction validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

loso_reconstruction_results = []

MIN_COMPOUNDS = 5

eligible_sources = [sid for sid, cpds in source_cpds.items() if len(cpds) >= MIN_COMPOUNDS]

for sid_true in tqdm(eligible_sources, desc="LOSO reconstruction"):
    true_compounds = set(source_cpds[sid_true])

    # Remove true source from candidate database
    database_minus_one = {sid: cpds for sid, cpds in source_cpds.items() if sid != sid_true}

    U_set = true_compounds
    A = []
    source_index = []

    for sid, compounds in database_minus_one.items():
        elems = compounds & U_set
        if elems:
            A.append(elems)
            source_index.append(sid)

    if not A:
        continue
 
    P = make_costfun(A=A, source_index=source_index, source_meta_lookup=source_meta_lookup, household_id=None)

    # Recoverable chemistry only
    union_A = set().union(*A)
    recoverable_compounds = true_compounds & union_A

    if len(recoverable_compounds) < MIN_COMPOUNDS:
        continue

    recoverable_U = sorted(recoverable_compounds)

    x, _ = greedy_cover(A, recoverable_U, P, source_index, alpha=best_alpha)

    recovered_sources = {source_index[i] for i in np.where(x == 1)[0]}
    recovered_compounds = set()

    for sid in recovered_sources:
        recovered_compounds.update(source_cpds[sid])

    # The full chemistry of the selected substitute sources is compared with the true source:
    # - precision: fraction of compounds in the substitute sources that also belong to the true source
    # - recall: fraction of the true source's compounds explained by the substitutes
    intersection = len(true_compounds & recovered_compounds)

    compound_precision = (
        intersection / len(recovered_compounds)
        if recovered_compounds else 0
    )

    compound_recall = (
        intersection / len(true_compounds)
        if true_compounds else 0
    )

    compound_f1 = (
        2 * compound_precision * compound_recall /
        (compound_precision + compound_recall)
        if (compound_precision + compound_recall) > 0
        else 0
    )

    jaccard = (
        intersection / len(true_compounds | recovered_compounds)
        if len(true_compounds | recovered_compounds) > 0
        else 0
    )

    expansion_factor = (
        len(recovered_compounds) / len(true_compounds)
        if true_compounds else np.nan
    )

    loso_reconstruction_results.append({
        "source_id": sid_true,
        "compound_precision": compound_precision,
        "compound_recall": compound_recall,
        "compound_f1": compound_f1,
        "jaccard": jaccard,
        "expansion_factor": expansion_factor,
        "n_true_compounds": len(true_compounds),
        "n_recoverable_compounds": len(recoverable_compounds),
        "n_recovered_compounds": len(recovered_compounds),
        "recoverable_fraction": len(recoverable_compounds) / len(true_compounds),
        "n_selected_sources": len(recovered_sources),
        "selected_sources": sorted(recovered_sources)
    })

loso_reconstruction_df = pd.DataFrame(loso_reconstruction_results)

summary_loso = (
    loso_reconstruction_df[
        [
            "compound_precision",
            "compound_recall",
            "compound_f1",
            "jaccard",
            "expansion_factor",
            "recoverable_fraction",
            "n_selected_sources",
            "n_true_compounds",
            "n_recoverable_compounds",
            "n_recovered_compounds",
        ]
    ]
    .mean()
    .rename({
        "n_selected_sources": "mean_selected_sources",
        "n_true_compounds": "mean_true_compounds",
        "n_recoverable_compounds": "mean_recoverable_compounds",
        "n_recovered_compounds": "mean_recovered_compounds",
    })
    .to_frame(name="Mean")
)

minorsep()
logging.info(f"LOSO reconstruction summary:")
display(summary_loso.round(3))

minorsep()
logging.info(f"LOSO reconstruction distribution:")
display(
    loso_reconstruction_df[
        [
            "compound_precision",
            "compound_recall",
            "compound_f1",
            "jaccard",
            "expansion_factor",
            "recoverable_fraction",
        ]
    ].describe().round(3)
)

minorsep()
logging.info(f"Worst reconstructed sources:")
display(
    loso_reconstruction_df.sort_values("compound_f1").head(10)
)

minorsep()
logging.info(f"Best reconstructed sources:")
display(
    loso_reconstruction_df.sort_values(
        "compound_f1",
        ascending=False
    ).head(10)
)

minorsep()
logging.info(f"Done running LOSO reconstruction validation | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Validation interpretation engine
majorsep()
logging.info(f"Defining validation interpretation engine | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

METRIC_INFO = {
# Recovery
"compound_precision": {
    "direction": "high",
    "excellent": 0.90,
    "good": 0.75,
    "moderate": 0.50,
    "measure": "Fraction of recovered compounds that truly belong to the target sample.",
    "excellent_text": "Recovered compounds are highly specific with very few false positives.",
    "good_text": "Most recovered compounds are correct with limited contamination.",
    "moderate_text": "Recovery is informative but includes noticeable false positives.",
    "weak_text": "Recovered compounds contain many incorrect assignments."
},

"compound_recall": {
    "direction": "high",
    "excellent": 0.90,
    "good": 0.75,
    "moderate": 0.50,
    "measure": "Fraction of true compounds successfully recovered.",
    "excellent_text": "Nearly all true compounds are recovered.",
    "good_text": "Most true compounds are recovered.",
    "moderate_text": "Only part of the chemistry is recovered.",
    "weak_text": "Large fractions of true chemistry are missed."
},

"compound_f1": {
    "direction": "high",
    "excellent": 0.90,
    "good": 0.75,
    "moderate": 0.50,
    "measure": "Balanced measure combining compound precision and recall.",
    "excellent_text": "Recovery is simultaneously accurate and complete.",
    "good_text": "Recovery is consistently strong.",
    "moderate_text": "Recovery is acceptable but imperfect.",
    "weak_text": "Recovery quality is poor."
},

"source_precision": {
    "direction": "high",
    "excellent": 0.80,
    "good": 0.60,
    "moderate": 0.40,
    "measure": "Fraction of inferred sources that are truly correct.",
    "excellent_text": "Few incorrect products are proposed.",
    "good_text": "Most inferred products are correct.",
    "moderate_text": "Recovered sources include several false positives.",
    "weak_text": "Many inferred sources are incorrect."
},

"source_recall": {
    "direction": "high",
    "excellent": 0.80,
    "good": 0.60,
    "moderate": 0.40,
    "measure": "Fraction of true sources successfully identified.",
    "excellent_text": "Nearly all generating sources are recovered.",
    "good_text": "Most generating sources are recovered.",
    "moderate_text": "Only part of the generating sources are recovered.",
    "weak_text": "Most generating sources remain unidentified."
},

"source_f1": {
    "direction": "high",
    "excellent": 0.80,
    "good": 0.60,
    "moderate": 0.40,
    "measure": "Balanced source-level recovery.",
    "excellent_text": "Excellent overall source identification.",
    "good_text": "Reliable source identification.",
    "moderate_text": "Moderately accurate source recovery.",
    "weak_text": "Poor source recovery."
},

"jaccard": {
    "direction": "high",
    "excellent": 0.80,
    "good": 0.60,
    "moderate": 0.40,
    "measure": "Similarity between recovered and true compound sets.",
    "excellent_text": "Recovered chemistry almost perfectly matches the truth.",
    "good_text": "Recovered chemistry strongly overlaps the truth.",
    "moderate_text": "Moderate overlap exists.",
    "weak_text": "Recovered chemistry differs substantially from the truth."
},

"false_discovery_rate": {
    "direction": "low",
    "excellent": 0.05,
    "good": 0.15,
    "moderate": 0.30,
    "measure": "Fraction of inferred sources that are incorrect.",
    "excellent_text": "Very few false discoveries occur.",
    "good_text": "False discoveries remain limited.",
    "moderate_text": "False discoveries are noticeable.",
    "weak_text": "False discoveries dominate the inferred solution."
},

"cost_ratio": {
    "direction": "target1",
    "excellent": 0.05,
    "good": 0.10,
    "moderate": 0.20,
    "measure": "Agreement between inferred Bayesian cost and true Bayesian cost.",
    "excellent_text": "Recovered solution closely matches the optimal Bayesian solution.",
    "good_text": "Recovered solution has similar Bayesian plausibility.",
    "moderate_text": "Recovered solution deviates somewhat from the optimum.",
    "weak_text": "Recovered solution differs substantially from the expected optimum."
},

"posterior_entropy": {
    "direction": "low",
    "excellent": 0.5,
    "good": 1.0,
    "moderate": 2.0,
    "measure": "Overall uncertainty across candidate explanations.",
    "excellent_text": "Posterior is highly concentrated around a few explanations.",
    "good_text": "Posterior shows limited ambiguity.",
    "moderate_text": "Posterior remains moderately diffuse.",
    "weak_text": "Posterior probability is spread over many explanations."
},

"effective_sample_size": {
    "direction": "low",
    "excellent": 2,
    "good": 5,
    "moderate": 10,
    "measure": "Effective number of equally probable explanations.",
    "excellent_text": "Very few explanations dominate the posterior.",
    "good_text": "Posterior concentrates on a small number of explanations.",
    "moderate_text": "Posterior remains moderately dispersed.",
    "weak_text": "Many explanations contribute similarly."
},

"top1_posterior": {
    "direction": "high",
    "excellent": 0.70,
    "good": 0.50,
    "moderate": 0.30,
    "measure": "Posterior probability assigned to the highest-ranked explanation.",
    "excellent_text": "One explanation clearly dominates.",
    "good_text": "A preferred explanation exists.",
    "moderate_text": "Several competing explanations remain.",
    "weak_text": "No dominant explanation exists."
},

"top5_posterior_mass": {
    "direction": "high",
    "excellent": 0.95,
    "good": 0.85,
    "moderate": 0.70,
    "measure": "Posterior probability contained within the five best explanations.",
    "excellent_text": "Nearly all posterior mass is concentrated in five solutions.",
    "good_text": "Most posterior mass is concentrated in five solutions.",
    "moderate_text": "Posterior remains moderately distributed.",
    "weak_text": "Posterior is widely dispersed."
},

"recoverable_fraction": {
    "direction": "high",
    "excellent": 0.90,
    "good": 0.75,
    "moderate": 0.50,
    "measure": "Fraction of compounds theoretically reconstructable after removing the true source.",
    "excellent_text": "Database redundancy allows nearly complete reconstruction.",
    "good_text": "Most chemistry remains reconstructable.",
    "moderate_text": "Only partial reconstruction is possible.",
    "weak_text": "Limited reconstruction is achievable."
},

"best_jaccard": {
    "direction": "low",
    "excellent": 0.10,
    "good": 0.20,
    "moderate": 0.40,
    "measure": "Similarity between database sources during LOSO analysis.",
    "excellent_text": "Database sources are highly distinct.",
    "good_text": "Only moderate redundancy exists.",
    "moderate_text": "Database contains noticeable redundancy.",
    "weak_text": "Database sources strongly overlap."
}
}

def interpret_metric(metric, value):
    info = METRIC_INFO[metric]

    if info["direction"] == "high":
        score = value
    elif info["direction"] == "low":
        score = -value
    else:
        score = -abs(value - 1)

    if info["direction"] == "target1":
        d = abs(value - 1)

        if d <= info["excellent"]:
            return "Excellent", info["excellent_text"], info["measure"]

        if d <= info["good"]:
            return "Good", info["good_text"], info["measure"]

        if d <= info["moderate"]:
            return "Moderate", info["moderate_text"], info["measure"]

        return "Weak", info["weak_text"], info["measure"]

    if score >= (info["excellent"] if info["direction"] == "high" else -info["excellent"]):
        return "Excellent", info["excellent_text"], info["measure"]

    if score >= (info["good"] if info["direction"] == "high" else -info["good"]):
        return "Good", info["good_text"], info["measure"]

    if score >= (info["moderate"] if info["direction"] == "high" else -info["moderate"]):
        return "Moderate", info["moderate_text"], info["measure"]

    return "Weak", info["weak_text"], info["measure"]

logging.info(f"Done defining validation interpretation engine | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Build master validation summary
majorsep()
logging.info(f"Building master validation summary | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

validation_rows = []

def add_metric(category, metric_name, metric_key, value):
    interpretation, explanation, measure = interpret_metric(metric_key, value)

    validation_rows.append({
        "Category": category,
        "Metric": metric_name,
        "Value": value,
        "Interpretation": interpretation,
        "Interpretation Explanation": explanation,
        "Metric Description": measure
    })

# Synthetic perturbation robustness
add_metric(
    "Synthetic perturbation robustness",
    "Compound precision",
    "compound_precision",
    perturb_df["precision"].mean()
)

add_metric(
    "Synthetic perturbation robustness",
    "Compound recall",
    "compound_recall",
    perturb_df["recall"].mean(),
)

# Synthetic chemistry recovery
add_metric(
    "Synthetic recovery",
    "Compound F1",
    "compound_f1",
    synth_validation_df["compound_f1"].mean()
)

add_metric(
    "Synthetic recovery",
    "Jaccard similarity",
    "jaccard",
    synth_validation_df["jaccard"].mean()
)

# Overlap sensitivity
add_metric(
    "Source ambiguity",
    "Source F1",
    "source_f1",
    overlap_validation_df["source_f1"].mean()
)

# Cost validation
add_metric(
    "Cost model",
    "Cost ratio",
    "cost_ratio",
    cost_df["cost_ratio"].mean()
)

# Beam search
add_metric(
    "Beam search",
    "Posterior entropy",
    "posterior_entropy",
    beam_validation_df["posterior_entropy"].mean()
)

add_metric(
    "Beam search",
    "Effective sample size",
    "effective_sample_size",
    beam_validation_df["effective_sample_size"].mean()
)

add_metric(
    "Beam search",
    "Top-1 posterior",
    "top1_posterior",
    beam_validation_df["top1_posterior"].mean()
)

add_metric(
    "Beam search",
    "Top-5 posterior mass",
    "top5_posterior_mass",
    beam_validation_df["top5_posterior_mass"].mean()
)

# LOSO redundancy
add_metric(
    "Database redundancy",
    "Maximum Jaccard",
    "best_jaccard",
    loso_df["best_jaccard"].mean()
)

# LOSO reconstruction
add_metric(
    "Algorithm robustness",
    "Recoverable fraction",
    "recoverable_fraction",
    loso_reconstruction_df["recoverable_fraction"].mean()
)

add_metric(
    "Algorithm robustness",
    "Compound F1",
    "compound_f1",
    loso_reconstruction_df["compound_f1"].mean()
)

master_validation_df = pd.DataFrame(validation_rows)

VAL_DIR = Path(DATA_DIR / "validation_outputs")
VAL_DIR.mkdir(exist_ok=True)
master_validation_df.to_csv(VAL_DIR / "master_validation_summary.csv", index=False)

logging.info(f"Master validation summary saved to: \n{VAL_DIR}")
minorsep()
logging.info(f"Done building master validation summary | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Convert sample_universes DF to dictionary
majorsep()
logging.info(f"Convert sample_universes DF to dictionary | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

sample_universes = {
    col: sample_universes_df[col].dropna().tolist()
    for col in sample_universes_df.columns
    if col != "inchikey"
}

logging.info(f" - n sample_universes samples: {len(sample_universes)}")
minorsep()
logging.info(f"Converted sample_universes DF to dictionary | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# OPTIONAL: Specify best alphas manually based on the plot
majorsep()

# best_alpha = 4500
# best_alpha_beam = 4500

logging.info(f"best_alpha to be used:      {best_alpha.round(2)}")
logging.info(f"best_alpha_beam to be used: {best_alpha_beam.round(2)}")
majorsep()


In [ ]:
# Per-sample cover analysis
majorsep()
logging.info(f"Running per-sample cover analysis | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

cover_start_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
results = {}

# Covers are ranked with the same cost the optimisers minimise:
#  - source_cost = P (prior/coverage cost) + alpha * frequency penalty
if best_alpha != best_alpha_beam:
    logging.warning(f" - best_alpha ({best_alpha}) != best_alpha_beam ({best_alpha_beam}); covers are ranked with best_alpha")
    minorsep()

samples_without_geo_metadata = [] # samples not found in household_metadata_priors.csv

for sample, U_raw in tqdm(sample_universes.items(), desc="Samples"):
    household_id = household_key(sample) # 'FR01_LS2' → 'FR01'
    logging.info(f"Sample: {sample} | Household: {household_id} | Initial |U|: {len(U_raw):,}")
    
    # Geo priors silently fall back to 1.0 if the sample ID is not in the geo metadata file
    if USE_GEO_PRIORS and household_id not in hh_metadata_lookup:
    # if household_id not in hh_metadata_lookup:
        samples_without_geo_metadata.append(household_id)

    # Restrict universe to compounds present in masked the source database
    U = sorted(set(U_raw).intersection(sources_masked["inchikey"]))

    # Report how many compounds cannot be attributed to any source
    n_unattributed = len(set(U_raw)) - len(U)
    logging.info(f" - Compounds without any source in the DB (unattributed): {n_unattributed:,} ({percent(n_unattributed, len(set(U_raw)))})")

    if len(U) == 0:
        logging.warning(f" - No overlap → skipping")
        continue

    U_set = set(U)
    A, source_index, P = build_problem(U_set, household_id=household_id)
    eq_map = None
    
    if A is None:
        logging.warning(f" - No candidate products/sources → skipping")
        continue

    assert len(A) == len(source_index), "len(A) is not equal to len(source_index)"
    assert len(P) == len(A), "Cost vector length mismatch"

    logging.info(f" - Candidate sources | Sets |A|: {len(A):,}")

    # Coverage check
    union_A = set.union(*A)
    # Missing
    missing = set(U) - union_A

    if missing:
        logging.warning(f" - Missing coverage: {len(missing):,} compounds")
        continue

    # Full per-source cost (identical to what greedy_cover uses internally)
    source_cost = calculate_source_costs(source_index, U_set, P, best_alpha)

    # GREEDY
    x, e = greedy_cover(A, U, P, source_index, alpha=best_alpha)
    greedy_sets = np.where(x == 1)[0].tolist()

    logging.info(f" - GREEDY sets: {len(greedy_sets):,}")

    # Beam Search
    # sample_seed = GLOBAL_SEED + hash(sample) % 10_000
    # (hash() of a string changes between Python sessions → not reproducible; the seed is not used inside beam search anyway)
    # narrower beam_width (cover 1 = GREEDY cover in ~all samples, see final checks) → roughly 4x faster
    T_tree = beam_search_covers(A, U, P, source_index, alpha=best_alpha_beam, beam_width=5, max_covers=50, 
                                diversity_lambda=0.25, seed=GLOBAL_SEED)
    
    # Combine GREEDY and Beam Search solutions
    T = [(greedy_sets, e)] + [(x, e) for x, e, _ in T_tree]

    # remove duplicate covers (same set indices)
    seen = set()
    T_unique = []

    assert len(T) > 0, "No valid covers generated"

    for item in T:
        x_set = item[0]
        e_vec = item[1]
        
        key = tuple(sorted(x_set))
        
        if key not in seen:
            seen.add(key)
            T_unique.append((x_set, e_vec))
    
    T = T_unique
    T.sort(key=lambda t: (source_cost[t[0]].sum(), len(t[0]))) # rank by full cost

    cover_costs = [source_cost[x_set].sum() for x_set, _ in T]

    if len(cover_costs) > 1:
        logging.info(f" - Cover cost variance: {np.var(cover_costs):.2f}")

    results[sample] = {"covers": T,
        "cover_costs": cover_costs,
        "A": A,
        "U": U,
        "sources": source_index,
        "eq_map": eq_map,
        "P": P,
        "source_cost": source_cost,
        "n_unattributed": n_unattributed,
        "alpha_greedy": best_alpha,
        "alpha_beam": best_alpha_beam}

    logging.info(f" - Total unique covers retained: {len(T):,}")
    majorsep()

logging.info(f"Samples without geo metadata (geo priors NOT applied): \n{len(samples_without_geo_metadata):,} of {len(sample_universes):,}")
if samples_without_geo_metadata:
    logging.warning(f"   Examples: {samples_without_geo_metadata[:10]}")
minorsep()

logging.info(f"Start per-sample cover analysis | {cover_start_time}")
logging.info(f"Done per-sample cover analysis | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Post-processing — costs & chemicals
majorsep()
logging.info(f"Post-processing all samples | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Initialize output containers
rows_sets = []
rows_costs = []

# Loop through all samples & extract objects
for sample, data in results.items():
    T = data["covers"] # all covers for this sample
    A = data["A"] # all candidate source sets
    P = data["P"] # source costs; use stored cost vector
    source_cost = data["source_cost"] # full cost (P + alpha * penalty) used by the optimisers

    # loop through covers & compute total cover cost
    for cid, item in enumerate(T, start=1):
        x_set = item[0]
        e_vec = item[1]

        # cost = item[2] if len(item) > 2 else sum(P[i] for i in x_set) # use stored cost
        # (item[2] never existed, so this always used P only, without the penalty)
        cost = float(sum(source_cost[i] for i in x_set))

        # source IDs in current cover
        source_ids = [data["sources"][i] for i in x_set]
        # number of unique products/sources
        n_unique_sources = len(set(source_ids))
        # collect all chemicals covered by this cover
        covered_chemicals = set()

        for i in x_set:
            covered_chemicals.update(A[i])
        
        n_unique_chemicals = len(covered_chemicals)

        # get the metadata rows of this cover's sources once, then count each label column
        if "source_id" in source_meta_lookup.columns:
            cover_meta = source_meta_lookup[source_meta_lookup["source_id"].isin(source_ids)]
        else:
            cover_meta = source_meta_lookup.loc[source_ids]

        source_values = cover_meta["source_harmonized"].dropna().unique() # same result as before

        n_unique_source_harmonized = len(source_values)
        n_source_classes = cover_meta["source_class"].dropna().nunique() # number of distinct source classes ("Domain | Type")
        n_source_class_domains = cover_meta["source_class_domain"].dropna().nunique() # number of distinct source class domains
        
        # store source membership rows
        for s in x_set:
            rows_sets.append({"sample": sample, "cover_id": cid, "set": s})

        # store total cover cost
        rows_costs.append({
            "sample": sample,
            "cover_id": cid,
            "cost": cost,
            "n_unique_chemicals": n_unique_chemicals,
            "n_unique_sources": n_unique_sources,
            "n_unique_source_harmonized": n_unique_source_harmonized,
            "n_source_classes": n_source_classes,
            "n_source_class_domains": n_source_class_domains
        })

# Convert to DataFrames
cover_sets = pd.DataFrame(rows_sets) # which sources/products belong to each cover
cover_costs = pd.DataFrame(rows_costs) # total cost of each cover

logging.info(f" - Total covers: {len(cover_costs):,}")

minorsep()
logging.info(f"Done post-processing | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Map set index → product/source ID
majorsep()
logging.info(f"Mapping set index → product/source ID | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

rows_sources = []

cover_sets_grouped = {sample: df for sample, df in cover_sets.groupby("sample")}

for sample, data in tqdm(results.items(), desc="Mapping samples → products/sources"):
    source_ids = data["sources"]
    P_vec = np.asarray(data["P"], dtype=float) # per-source cost = -log p(source) from make_costfun
    A = data["A"] # candidate source sets (compounds of each source found in this sample)
    n_U = max(len(data["U"]), 1) # number of sample compounds with a source in the DB
    household_id = household_key(sample) # same household key as in the set cover loop
    prior_cache = {} # each source's prior is computed only once per sample

    for row in cover_sets_grouped.get(sample, []).itertuples(index=False):
        set_idx = int(row.set)

        if set_idx < len(source_ids):
            sid = source_ids[set_idx]
            if set_idx not in prior_cache:
                prior_cache[set_idx] = float(source_prior(sid, household_id=household_id))

            rows_sources.append({
                "sample": sample,
                "cover_id": row.cover_id,
                "source_id": sid,
                "source_prior": prior_cache[set_idx], # same prior as used in make_costfun
                "source_coverage": len(A[set_idx]) / n_U, # fraction of sample compounds in this source
                "_neglog_P": float(P_vec[set_idx]) # temporary helper (= -ln source_P)
            })

cover_sources = pd.DataFrame(rows_sources)

# Per-source probabilities on a log10 scale
#  - With PRIOR_STRENGTH ≈ 65, p(source) = prior^65 × coverage spans hundreds of orders of magnitude, so
#    exp(-P_i) underflows to exactly 0 for most sources (numbers below ~1e-308 cannot be stored)
#  - source_log10P = log10 of the model's per-source probability (normalized over all candidate sources
#    of the sample); e.g. -238.3 means 4.5e-239; never underflows
#  - source_log10P_rel = source_log10P minus the highest source_log10P in the same cover
#    0 = strongest source of the cover; -10 = 1e10 times less likely than the strongest
if not cover_sources.empty:
    cover_sources["source_log10P"] = -cover_sources["_neglog_P"] / np.log(10)

    grp_keys = [cover_sources["sample"], cover_sources["cover_id"]]
    cover_sources["source_log10P_rel"] = (cover_sources["source_log10P"]
        - cover_sources.groupby(grp_keys)["source_log10P"].transform("max"))

    cover_sources = cover_sources.drop(columns="_neglog_P")

    logging.info(f" - Added source_prior, source_coverage, source_log10P, source_log10P_rel to cover_sources")

minorsep()
logging.info(f"Done mapping set index → product/source ID | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Convert costs to probabilities
majorsep()
logging.info(f"Computing probabilities (per sample) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# - T must be on the scale of the cost GAPS between covers (hundreds–thousands here), not ~1: a small T
#   makes the best cover take all probability (P = 1, all others 0)
# - "auto" → T = median over samples of the per-sample median gap (cost - best cost) of the alternative
#   covers; the typical alternative cover then gets exp(-1) ≈ 37 % of the best cover's weight
# - or set a number, e.g., POSTERIOR_TEMPERATURE_SETTING = 500.0
# - POSTERIOR_TEMPERATURE (the number actually used) is also used by the "Build posterior over covers" cell
POSTERIOR_TEMPERATURE_SETTING = "auto" # "auto" or a number

if cover_costs.empty:
    logging.warning(f" - WARNING: No costs to normalize")
    minorsep()
else:
    cover_costs = cover_costs.copy()

    # cost gaps between each cover and the best cover of the same sample
    cost_gap = cover_costs["cost"] - cover_costs.groupby("sample")["cost"].transform("min")
    alt_gaps = cost_gap[cost_gap > 0]

    if len(alt_gaps) > 0:
        q = alt_gaps.quantile([0.05, 0.25, 0.5, 0.75, 0.95])
        logging.info(f" - Cost gaps of alternative covers vs. best cover (all samples): "
                     f"\n   5%: {q[0.05]:,.1f} | 25%: {q[0.25]:,.1f} | Median: {q[0.5]:,.1f} | "
                     f"75%: {q[0.75]:,.1f} | 95%: {q[0.95]:,.1f}")

    if POSTERIOR_TEMPERATURE_SETTING == "auto":
        per_sample_median_gap = alt_gaps.groupby(cover_costs.loc[alt_gaps.index, "sample"]).median()
        if len(per_sample_median_gap) > 0:
            POSTERIOR_TEMPERATURE = float(per_sample_median_gap.median())
        else:
            POSTERIOR_TEMPERATURE = 1.0 # only one cover per sample → temperature has no effect
            logging.warning(f" - No alternative covers found → POSTERIOR_TEMPERATURE set to 1.0")
    else:
        POSTERIOR_TEMPERATURE = float(POSTERIOR_TEMPERATURE_SETTING)

    logging.info(f" - POSTERIOR_TEMPERATURE used: {POSTERIOR_TEMPERATURE:,.2f} "
                 f"(setting: {POSTERIOR_TEMPERATURE_SETTING})")

    # stable exponentiation
    cover_costs["P"] = cover_costs.groupby("sample")["cost"].transform(
        lambda c: np.exp(-(c - c.min()) / POSTERIOR_TEMPERATURE))

    # normalize per sample
    # prevents divide-by-zero edge cases
    cover_costs["P"] = cover_costs.groupby("sample")["P"].transform(
        lambda x: x / (x.sum() + 1e-12))

    assert ((cover_costs["P"] >= 0) & (cover_costs["P"] <= 1)).all(), \
        "Posterior probabilities outside [0,1]"
    
    assert np.allclose(
        cover_costs.groupby("sample")["P"].sum().values,
        1.0,
        atol=1e-8
    ), "Posterior probabilities do not sum to one"

    # How concentrated is the probability now?
    best_P = cover_costs.groupby("sample")["P"].max()
    logging.info(f" - Samples where the best cover has P > 0.99: {(best_P > 0.99).sum():,} of {len(best_P):,} | "
                 f"Median best-cover P: {best_P.median():.3f}")

minorsep()
logging.info(f"Probability normalization (per sample) complete | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Marginal product/source probabilities
majorsep()
logging.info(f"Computing marginal product/source probabilities | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

if cover_sets.empty or cover_costs.empty:
    logging.warning(f" - Skipping marginal probability calculation")
else:
    # convert cost → probability
    cover_costs = cover_costs.copy()

    # join to products/sources directly
    merged = cover_sources.merge(
        cover_costs[["sample", "cover_id", "P"]],
        on=["sample", "cover_id"],
        how="left"
    )
    
    # marginal per source_id
    marginal = (merged.groupby(["sample", "source_id"])
        .agg(marginal_P=("P", "sum"), # defines source-level probability mass
            count=("P", "size")) # how many covers contributed
        .reset_index())

    logging.info(f" - Products/sources with non-zero probability: {len(marginal):,}")

minorsep()
logging.info(f"Done computing marginal product/source probabilities | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Probabilistic Graphical Model (PGM) interpretation
majorsep()
logging.info(f"Constructing latent source posterior model (PGM view) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

"""
The system is reinterpreted as a latent-variable model:
- Observed: U (set of detected compounds per sample)
- Latent: S ⊆ sources (active emitting sources)
- Generative intuition:
  - Each source s emits a compound set C_s
  - Observed U is a union over active sources (with noise/masking)

Approximate:
- P(S | U) ∝ exp(- Energy(S, U) / T)
- Energy(S, U) = sum of full source costs (P + alpha * penalty), i.e. the objective the optimisers minimise
- posterior_P = marginal inclusion probability of a source = sum of weights of all covers containing it
                (in [0, 1]; values of different sources do NOT sum to 1)
- posterior_share = posterior_P renormalized to sum to 1 per sample (share of the explanation)
"""

# - Extra compounds (compounds of a source not found in the sample) are already penalized through
#   alpha * penalty inside source_cost, so they are not added a second time here
def cover_energy(x_set, source_cost):
    return float(sum(source_cost[i] for i in x_set))

# Build posterior over covers
posterior_rows = []

for sample, data in results.items():
    T = data["covers"]
    A = data["A"]
    P = data["P"]
    source_cost = data["source_cost"]
    U_set = set(data["U"])

    # compute unnormalized log posterior weights
    energies = []
    
    for item in T:
        x_set = item[0]
        E = cover_energy(x_set, source_cost)
        energies.append(E)

    energies = np.array(energies)

    # convert to log-probabilities (stable softmax)
    log_w = - (energies - np.min(energies)) / POSTERIOR_TEMPERATURE # POSTERIOR_TEMPERATURE defined in "Convert costs to probabilities" cell
    w = np.exp(log_w)
    w = w / (np.sum(w) + 1e-12)

    # attach posterior weight per cover
    for item, pw in zip(T, w):
        x_set = item[0]

        for i in x_set:
            posterior_rows.append({
                "sample": sample,
                "source_id": data["sources"][i],
                "posterior_weight": pw
            })

posterior_df = pd.DataFrame(posterior_rows)

# True marginal posterior over sources
source_posterior = (
    posterior_df
    .groupby(["sample", "source_id"])
    .agg(
        posterior_P=("posterior_weight", "sum"),
        cover_support=("posterior_weight", "size")
    )
    .reset_index()
)

# posterior_P is now kept as the inclusion probability (not renormalized)
# The renormalized version (share of the explanation per sample) is stored separately
source_posterior["posterior_P"] = source_posterior["posterior_P"].clip(upper=1.0) # removes tiny rounding above 1
source_posterior["posterior_share"] = source_posterior.groupby("sample")["posterior_P"].transform(lambda x: x / (x.sum() + 1e-12))

# Diagnostics
logging.info(f"Posterior model constructed:")
logging.info(f" - samples: {posterior_df['sample'].nunique():,}")
logging.info(f" - unique sources: {posterior_df['source_id'].nunique():,}")
logging.info(f" - rows: {len(posterior_df):,}")
minorsep()

display(source_posterior.round(4))

# Saving files
OUT_DIR = Path(DATA_DIR / "set_cover_outputs")
OUT_DIR.mkdir(exist_ok=True)

# Global file (CSV only, Excel is too large)
source_posterior.to_csv(OUT_DIR / "source_posterior_all_samples_beam.csv", index=False)
minorsep()
logging.info(f"Results saved to: \n{OUT_DIR}")

minorsep()
logging.info(f"Done constructing latent source posterior model | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Posterior diagnostics
majorsep()
logging.info(f"Computing posterior diagnostics | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

posterior_rows = []
robustness_rows = []

for sample, grp in merged.groupby("sample"):
    posterior = (
        source_posterior
        .loc[source_posterior["sample"] == sample, "posterior_P"]
        .values
    )

    posterior = posterior[posterior > 0]

    if len(posterior) == 0:
        continue

    posterior = posterior / posterior.sum()
    entropy = -np.sum(posterior * np.log(posterior + 1e-12))
    ess = 1.0 / np.sum(posterior ** 2)
    top1 = posterior.max()
    top5 = np.sort(posterior)[-5:].sum()
    concentration = top1 / max(top5, 1e-12)
    p_sorted = np.sort(posterior)
    n = len(p_sorted)

    gini = (2 * np.sum((np.arange(1, n + 1)) * p_sorted) / (n * p_sorted.sum())) - (n + 1) / n
    total_covers = grp["cover_id"].nunique()
    total_weight = grp["P"].sum()
    source_freq = []

    for sid, sub in grp.groupby("source_id"):
        n_present = sub["cover_id"].nunique()
        solution_frequency = n_present / total_covers
        weighted_frequency = sub["P"].sum() / (total_weight + 1e-12)
        alpha = n_present + 1
        beta_param = total_covers - n_present + 1
        ci_low = beta.ppf(0.025, alpha, beta_param)
        ci_high = beta.ppf(0.975, alpha, beta_param)

        robustness_rows.append({
            "sample": sample,
            "source_id": sid,
            "solution_frequency": solution_frequency,
            "weighted_frequency": weighted_frequency,
            "covers_present": n_present,
            "total_covers": total_covers,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "ci_width": ci_high - ci_low
        })

        source_freq.append(solution_frequency)

    posterior_rows.append({
        "sample": sample,
        "posterior_entropy": entropy,
        "posterior_ess": ess,
        "top1_posterior": top1,
        "top5_posterior_mass": top5,
        "posterior_concentration": concentration,
        "posterior_gini": gini,
        "n_covers": total_covers,
        "mean_solution_frequency": np.mean(source_freq),
        "median_solution_frequency": np.median(source_freq),
        "sd_solution_frequency": np.std(source_freq)
    })

posterior_summary_df = pd.DataFrame(posterior_rows)
robustness_df = pd.DataFrame(robustness_rows)

logging.info(f"Posterior summary:")
display(posterior_summary_df.head())
minorsep()
logging.info(f"Robustness summary:")
display(robustness_df.head())

posterior_summary_df.to_csv(VAL_DIR / "posterior_summary.csv", index=False)
robustness_df.to_csv(VAL_DIR / "robustness_summary.csv", index=False)
minorsep()
logging.info(f"Posterior summary & robustness summary saved to: \n{VAL_DIR}")

minorsep()
logging.info(f"Computed posterior diagnostics | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Do priors or the penalty drive the source choice?
majorsep()
logging.info(f"Checking influence of priors vs. frequency penalty | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

influence_rows = []
origin_rows = []

for sample, data in results.items():
    P = np.asarray(data["P"])
    penalty_term = np.asarray(data["source_cost"]) - P # = alpha * normalized penalty

    influence_rows.append({
        "sample": sample,
        "prior_cost_range_p5_p95": np.percentile(P, 95) - np.percentile(P, 5),
        "penalty_cost_range_p5_p95": np.percentile(penalty_term, 95) - np.percentile(penalty_term, 5),
        "share_sources_with_zero_penalty": float(np.mean(penalty_term == 0))
    })

    # origins of the sources in the best-ranked cover
    best_cover = data["covers"][0][0]
    for i in best_cover:
        sid = data["sources"][i]
        origin_rows.append({"sample": sample, "source_origin": source_metadata_lookup_dict[sid]["origin"]})

influence_df = pd.DataFrame(influence_rows)
logging.info(f"Cost ranges across candidate sources (median over samples):\n")
logging.info(influence_df[["prior_cost_range_p5_p95", "penalty_cost_range_p5_p95", "share_sources_with_zero_penalty"]].median().round(3))
minorsep()
logging.info(f"If prior_cost_range << penalty_cost_range, the priors (SOURCE_WEIGHT, categories, geo metadata) hardly influence the result")
minorsep()

origin_df = pd.DataFrame(origin_rows)
logging.info(f"Source origins in the best cover (all samples):\n")
logging.info(origin_df["source_origin"].value_counts(normalize=True).round(3))

minorsep()
logging.info(f"Done checking influence of priors vs. frequency penalty | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Which source origin explains how many compounds (best cover per sample)
majorsep()
logging.info(f"Computing compound attribution per source origin (best cover) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Each compound is attributed to the highest-priority origin (tier) among the sources of the best cover that contain it.
tier_attr_rows = []

for sample, data in results.items():
    A = data["A"]
    sources = data["sources"]
    best_cover = data["covers"][0][0]
    n_U = len(data["U"])

    best_cover_sorted = sorted(best_cover, key=lambda i: (ORIGIN_TO_TIER.get(source_metadata_lookup_dict[sources[i]]["origin"], N_TIERS - 1), i))

    claimed = set()
    counts = defaultdict(int)

    for i in best_cover_sorted:
        origin = source_metadata_lookup_dict[sources[i]]["origin"]
        new = A[i] - claimed
        counts[origin] += len(new)
        claimed |= new

    for origin, c in counts.items():
        tier_attr_rows.append({"sample": sample, "source_origin": origin, "n_compounds": c, "share_of_attributed_compounds": c / n_U if n_U else 0})

tier_attr_df = pd.DataFrame(tier_attr_rows)

tier_attr_wide = (tier_attr_df
    .pivot_table(index="sample", columns="source_origin", values="share_of_attributed_compounds", fill_value=0))

logging.info(f"Mean share of attributed compounds per origin (over samples):\n")
logging.info(tier_attr_wide.mean().sort_values(ascending=False).round(3))

OUT_DIR = Path(DATA_DIR / "set_cover_outputs")
OUT_DIR.mkdir(exist_ok=True)
tier_attr_df.to_csv(OUT_DIR / "compound_attribution_per_origin.csv", index=False)
minorsep()
logging.info(f"Saved compound_attribution_per_origin.csv to: \n{OUT_DIR}")

minorsep()
logging.info(f"Done computing compound attribution per source origin | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Final checks (own household products, GREEDY vs. best cover, QC columns, unattributed compounds)
majorsep()
logging.info(f"Running final checks | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Can the household of every own analyzed product be read from its ID (e.g., 'FR01_CP04_HHP' → 'FR01')?
hh_ids = sorted(sid for sid, m in source_metadata_lookup_dict.items() if m["origin"] == "HOUSEHOLD_PRODUCTS")
hh_unreadable = [sid for sid in hh_ids if hh_product_household(sid) is None]
hh_households = sorted({hh_product_household(sid) for sid in hh_ids} - {None})
logging.info(f"Own household products: {len(hh_ids):,} | Household readable from ID: {len(hh_ids) - len(hh_unreadable):,}\n")
logging.info(f"Households with analyzed products: \n{len(hh_households):,} → {hh_households}")
if hh_unreadable:
    logging.warning(f"  - NOT readable (treated as 'other country'): {hh_unreadable}")
minorsep()

# Per sample: own products, GREEDY vs. best cover (cover 1), unattributed compounds
check_rows = []

for sample, data in tqdm(results.items(), desc="Final checks"):
    hh = household_key(sample)
    sources = data["sources"]
    best_cover = set(int(i) for i in data["covers"][0][0])

    own_hh = {i for i, sid in enumerate(sources)
              if source_metadata_lookup_dict[sid]["origin"] == "HOUSEHOLD_PRODUCTS" and hh_product_household(sid) == hh}
    any_hh = {i for i, sid in enumerate(sources) if source_metadata_lookup_dict[sid]["origin"] == "HOUSEHOLD_PRODUCTS"}

    # GREEDY cover recomputed with the same inputs (fast; no beam search)
    x_g, _ = greedy_cover(data["A"], data["U"], data["P"], sources, alpha=best_alpha)
    greedy_set = set(np.where(x_g == 1)[0].tolist())

    n_U = len(data["U"])
    n_unattr = data.get("n_unattributed", np.nan)

    check_rows.append({
        "sample": sample,
        "household": hh,
        "household_has_analyzed_products": hh in hh_households,
        "n_own_hh_candidates": len(own_hh),
        "n_own_hh_in_best_cover": len(own_hh & best_cover),
        "n_any_hh_in_best_cover": len(any_hh & best_cover),
        "n_sources_in_best_cover": len(best_cover),
        "best_cover_is_greedy": greedy_set == best_cover,
        "n_covers": len(data["covers"]),
        "n_compounds_attributable": n_U,
        "n_compounds_unattributed": n_unattr,
        "share_unattributed": n_unattr / (n_unattr + n_U) if (n_U + (n_unattr if pd.notna(n_unattr) else 0)) else np.nan
    })

final_checks_df = pd.DataFrame(check_rows)

own = final_checks_df[final_checks_df["household_has_analyzed_products"]]
minorsep()
logging.info(f"Samples from households WITH analyzed products: {len(own):,} of {len(final_checks_df):,}")
if len(own):
    logging.info(f" - with >= 1 own product among candidates: {(own['n_own_hh_candidates'] > 0).sum():,}")
    logging.info(f" - with >= 1 own product in best cover: {(own['n_own_hh_in_best_cover'] > 0).sum():,}")
    logging.info(f" - mean share of own products among best-cover sources: {(own['n_own_hh_in_best_cover'] / own['n_sources_in_best_cover']).mean():.3f}")
other = final_checks_df[~final_checks_df["household_has_analyzed_products"]]
if len(other):
    logging.info(f"\nSamples from other households:\n - mean share of (other households') products in best cover: "
                 f"{(other['n_any_hh_in_best_cover'] / other['n_sources_in_best_cover']).mean():.3f}")
minorsep()

logging.info(f"Best cover (cover 1) identical to the GREEDY cover in {final_checks_df['best_cover_is_greedy'].mean():.1%} of samples")
logging.info(f" - Covers per sample: median {final_checks_df['n_covers'].median():.0f}, max {final_checks_df['n_covers'].max():.0f}")
minorsep()

qc_like = [s for s in final_checks_df["sample"] if "blank" in str(s).lower() or str(s).upper().startswith("7S_CS")]
logging.info(f"QC/blank columns analyzed as samples: \n{len(qc_like)} {qc_like[:20]}")
if qc_like:
    logging.warning(f"   → blanks/QC were NOT removed in preprocessing part")
minorsep()

logging.info(f"Share of compounds without any source in the database: \n - Median {final_checks_df['share_unattributed'].median():.2f} "
             f"(IQR {final_checks_df['share_unattributed'].quantile(0.25):.2f}-{final_checks_df['share_unattributed'].quantile(0.75):.2f})")

minorsep()
final_checks_df.to_csv(VAL_DIR / "final_checks_per_sample.csv", index=False)
logging.info(f"Saved final_checks_per_sample.csv to: \n{VAL_DIR}")

minorsep()
logging.info(f"Done running final checks | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# OPTIONAL: Own household products — are "their" compounds explained by them? (+ household factor check)
majorsep()
logging.info(f"Checking own household products in detail | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

own_rows = []

for sample, data in results.items():
    hh = household_key(sample)
    sources = data["sources"]
    A = data["A"]
    best_cover = set(int(i) for i in data["covers"][0][0])

    is_hhp = [source_metadata_lookup_dict[sid]["origin"] == "HOUSEHOLD_PRODUCTS" for sid in sources]
    own = [i for i, sid in enumerate(sources) if is_hhp[i] and hh_product_household(sid) == hh]
    other_hh = [i for i, sid in enumerate(sources) if is_hhp[i] and hh_product_household(sid) != hh]

    if not own:
        continue

    # compounds of this sample that occur in at least one own product
    own_cpds = set().union(*[A[i] for i in own])
    own_in_cover = [i for i in own if i in best_cover]
    explained_by_own = set().union(*[A[i] for i in own_in_cover]) if own_in_cover else set()

    # which origins in the best cover explain the own-product compounds NOT explained by own products?
    rest = own_cpds - explained_by_own
    origin_counts = defaultdict(int)
    for c in rest:
        for o in {source_metadata_lookup_dict[sources[i]]["origin"] for i in best_cover if c in A[i]}:
            origin_counts[o] += 1
    # main_other = max(origin_counts, key=origin_counts.get) if origin_counts else ""
    main_other = max(origin_counts, key=origin_counts.get) if origin_counts else "None (all explained by own products)"

    own_rows.append({
        "sample": sample,
        "n_own_candidates": len(own),
        "n_own_in_best_cover": len(own_in_cover),
        "n_sample_compounds_in_own_products": len(own_cpds),
        "share_explained_by_own_products": round(len(explained_by_own) / len(own_cpds), 3) if own_cpds else np.nan,
        "main_origin_explaining_the_rest": main_other,
        "n_rest_explained_by_main_origin": origin_counts.get(main_other, 0),
        # household factor actually applied (should be 2.0 (change to 1.0 etc. if needed) for own, 0.5 same country, 0.2 other country)
        "hh_factor_own_products": float(np.mean([hh_relation_factor(sources[i], hh) for i in own])),
        "hh_factor_other_hh_products": float(np.mean([hh_relation_factor(sources[i], hh) for i in other_hh])) if other_hh else np.nan,
    })

own_check_df = pd.DataFrame(own_rows)
display(own_check_df)

minorsep()
logging.info(f" - Samples with own products among candidates: {len(own_check_df):,}")
if len(own_check_df):
    logging.info(f" - Median share of 'own-product' sample compounds explained by own products: {own_check_df['share_explained_by_own_products'].median():.2f}")
    logging.info(f" - Household factor own products (should be {HH_SAME_HOUSEHOLD}): {own_check_df['hh_factor_own_products'].mean():.2f}")
    logging.info(f" - Household factor other households' products (0.2-0.5): {own_check_df['hh_factor_other_hh_products'].mean():.2f}")

own_check_df.to_csv(VAL_DIR / "own_household_products_check.csv", index=False)

minorsep()
logging.info(f"Done checking own household products | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Final DF & QC Summary
majorsep()
logging.info(f"Creating final DF & QC summary & saving output files | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Merge all outputs
final_df = (cover_sources
    .merge(cover_costs, on=["sample", "cover_id"], how="left", validate="many_to_one")
    .merge(marginal, on=["sample", "source_id"], how="left", validate="many_to_one")
    .merge(source_names, on="source_id", how="left", validate="many_to_one")
    .merge(source_lookup, on="source_id", how="left", validate="many_to_one"))

logging.info(f" - Initial final DF rows: {len(final_df):,}")

# Expand to compound-level
final_df = final_df.merge(source_compounds, on="source_id", how="left")

logging.info(f" - Rows after source_compounds merge: {len(final_df):,}")

# Add posterior_P
final_df = final_df.merge(source_posterior[["sample", "source_id", "posterior_P"]],
    on=["sample", "source_id"],
    how="left",
    validate="many_to_one")

logging.info(f" - Rows after source_posterior merge: {len(final_df):,}")

# Keep only compounds present in the specific sample
# Build fast lookup: sample → inchikeys
sample_to_inchikeys = {s: set(sample_universes[s]) for s in sample_universes}
valid_pairs = set((s, ik) for s, iks in sample_to_inchikeys.items() for ik in iks)

final_df = final_df[final_df.set_index(["sample", "inchikey"]).index.isin(valid_pairs)]

logging.info(f" - Rows after valid_pairs merge: {len(final_df):,}")

# Attach feature names
final_df = final_df.merge(dust_features,
    on="inchikey",
    how="left")

logging.info(f" - Rows after dust_features merge: {len(final_df):,}")

# Add statistics
final_df = final_df.merge(robustness_df,
    on=["sample", "source_id"],
    how="left",
    validate="many_to_one")

logging.info(f" - Rows after robustness merge: {len(final_df):,}")

final_df = final_df[[
    "sample",
    "cover_id",
    "source_id",
    "source_name",
    "source_origin",
    "source_kind",
    "source_general_category",
    "source_harmonized",
    "source_class",
    "source_class_domain",
    "cost",
    "P",
    "marginal_P",
    "posterior_P",
    "source_prior",
    "source_coverage",
    # "source_P",
    # "source_P_share",
    "source_log10P",
    "source_log10P_rel",
    "solution_frequency",
    "total_covers",
    "ci_low",
    "ci_high",
    "count",
    "inchikey",
    "feature_name",
    "n_unique_chemicals",
    "n_unique_sources",
    "n_unique_source_harmonized",
    "n_source_classes",
    "n_source_class_domains",
]]

logging.info(f" - Output final DF rows: {len(final_df):,}")
minorsep()

# Saving files
OUT_DIR = Path(DATA_DIR / "set_cover_outputs")
OUT_DIR.mkdir(exist_ok=True)

# Global file (CSV only, Excel is too large)
logging.info(f" - Saving global CSV file")
final_df.to_csv(OUT_DIR / "cover_results_all_samples_beam.csv", index=False)
logging.info(f" - Saved global CSV file to: \n{OUT_DIR}")
minorsep()


# Save all unattributed compounds per sample = compounds of a sample that do not appear in any cover
# of that sample in final_df
# These are mainly compounds without any source in the masked source database (= n_unattributed in the set cover loop) 
# plus all compounds of samples that were skipped (no overlap / no candidate sources)
logging.info(f" - Saving unattributed compounds per sample")

attributed_pairs = set(zip(final_df["sample"], final_df["inchikey"])) # (sample, inchikey) in any cover

unattributed_rows = [
    {"sample": s, "inchikey": ik}
    for s, iks in sample_to_inchikeys.items()
    for ik in sorted(iks, key=str)
    if pd.notna(ik) and (s, ik) not in attributed_pairs
]
unattributed_df = pd.DataFrame(unattributed_rows, columns=["sample", "inchikey"])

# Attach feature names (same lookup as for final_df)
unattributed_df = unattributed_df.merge(dust_features, on="inchikey", how="left")

unattributed_df = unattributed_df[[
    "sample",
    "inchikey",
    "feature_name"
]]

unattributed_df.to_csv(OUT_DIR / "unattributed_results_all_samples_beam.csv", index=False)

n_total_pairs = sum(len([ik for ik in iks if pd.notna(ik)]) for iks in sample_to_inchikeys.values())
n_unattr_pairs = unattributed_df[["sample", "inchikey"]].drop_duplicates().shape[0]
logging.info(f" - Unattributed compounds: {n_unattr_pairs:,} of {n_total_pairs:,} sample × compound pairs "
             f"({percent(n_unattr_pairs, n_total_pairs)}) in {unattributed_df['sample'].nunique():,} samples")
logging.info(f" - Saved unattributed_results_all_samples_beam.csv to: \n{OUT_DIR}")
minorsep()


# Identify sample columns
start_col = list(dust_full_std.columns).index("Concentration Type") + 1
sample_cols = dust_full_std.columns[start_col:]

# Per-sample files
for sample in tqdm(sample_cols, desc="Saving per-sample files"):
    df_s = final_df[final_df["sample"] == sample]

    if df_s.empty:
        continue

    safe_name = str(sample).replace(" ", "_").replace("/", "_") + "_beam"
    df_s.to_csv(OUT_DIR / f"{safe_name}.csv", index=False)

logging.info(f" - Saved per-sample CSV files to: \n{OUT_DIR}")
minorsep()

logging.info(f"All outputs saved to: \n{OUT_DIR}")
minorsep()

logging.info(f"Data per last sample:")
logging.info(f" - |U| (sample): {len(U):,}")
logging.info(f" - |A| (sources): {len(A):,}")
minorsep()
logging.info(f"Total covers generated: {len(cover_costs):,}")

minorsep()
logging.info(f"Created final DF & QC summary & saved output files | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()
logging.info(f"Pipeline complete | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()
